# Study B — Task 2 onwards (separate notebook)

Same pipeline as `KG_Strong_Pipeline.ipynb` with `STUDY = "B"` preset. Task 1 (section 5) is **not rerun**:
its results are read from `study_B/results/task1_runs.csv`, which the main notebook wrote when it finished
Task 1. Everything else (universe, embeddings, graph, prices, pairs, runs) loads from Drive when it exists.

**Run (Colab):** Runtime → Change runtime type → **GPU** (T4) → `Runtime → Run all`. After a crash or
disconnect, `Run all` again: finished stages and runs load from Drive.

## 0 · Setup

In [ ]:
# CELL 0.1 — install (Colab already ships torch; this adds the rest)
import os, sys, subprocess
SMOKE = os.environ.get("KG_SMOKE") == "1"   # local smoke test only; leave unset on Colab
if not SMOKE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentence-transformers>=3.0", "yfinance>=0.2.54", "torch_geometric>=2.5",
                    "python-igraph>=0.11", "leidenalg>=0.10", "xgboost>=2.0", "pyarrow>=15"], check=False)
print("install step done")

install step done


In [ ]:
# CELL 0.2 — configuration (the only cell you normally edit)
STUDY = "B"                                  # this notebook is for Study B
RUN_PROFILE = "full"                         # "full" = all models; "core" = pair-only + GAT + R-GCN only
RUN_BRIDGE = (STUDY == "A")                  # bridge test to the earlier draft design (Study A only)

import json, math, time, random, hashlib, pickle, itertools, warnings, re, shutil, zipfile, platform
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

CFG = dict(
    STUDY=STUDY,
    FINDB_COMMIT="e45fb93bd48b7b5a590280fb4c143c478120806e",   # committed 2021-07-23
    GRAPH_SNAPSHOT_DATE="2021-07-31",
    QUOTAS={
        "A": dict(Equity=1200, ETF=500, Fund=400, Crypto=300, Index=200, Currency=150),
        "B": dict(Equity=3000, ETF=1200, Fund=800, Crypto=600, Index=400, Currency=300),
    }[STUDY],
    PRIMARY_EQUITY_EXCHANGES=["NYQ", "NMS", "NGM", "NCM", "ASE", "LSE", "TOR", "PAR", "AMS", "GER",
                              "HKG", "NSE", "JPX", "ASX", "STO", "MIL", "MCE", "EBS", "KSC", "TAI",
                              "SAO", "JNB", "CPH", "OSL", "HEL", "BRU", "SES", "NZE"],
    MIN_SUMMARY_CHARS=40,
    TEXT_MODEL="sentence-transformers/all-mpnet-base-v2",
    K_SEM=15, ATTR_CAP=25,
    WINDOWS={"H0": ("2020-08-01", "2021-08-01"), "t1": ("2021-08-01", "2022-08-01"),
             "t2": ("2022-08-01", "2023-08-01"), "t3": ("2023-08-01", "2024-08-01"),
             "t4": ("2024-08-01", "2025-08-01")},
    PREV={"t1": "H0", "t2": "t1", "t3": "t2", "t4": "t3"},
    FOLDS=[dict(name="F1", train="t1", val="t2", test="t3"),
           dict(name="F2", train="t2", val="t3", test="t4")],
    DOWNLOAD_START="2020-07-01", DOWNLOAD_END="2025-08-10",
    ASSET_MIN_OBS_H0=120, MIN_OVERLAP_DAILY=180, MIN_OVERLAP_WEEKLY=35, LAG_MIN_OVERLAP=60,
    LAG_MIN_OVERLAP_WEEKLY=26,
    TAU_MAIN=0.50, TAU_SENS=[0.40, 0.60],
    SPLIT_FRACTIONS=(0.60, 0.20, 0.20), SPLIT_SEED=42,
    MODEL_SEEDS=[42, 7, 123, 2024, 99, 314, 777, 1001, 2025, 909],
    SENS_SEEDS=[42, 7, 123],
    HP_BUDGET=6,
    TASK1_MAX_EPOCHS=300, TASK1_PATIENCE=20,
    TASK2_MAX_EPOCHS=120, TASK2_PATIENCE=15,
    PAIR_BATCH=65536, GRAD_CLIP=5.0, WEIGHT_DECAY=1e-4, CKPT_EVERY=5,
    BOOTSTRAP_REPS={"A": 2000, "B": 1000}[STUDY],
    PERM_REPEATS=5,
    PORTFOLIO_SIZES=[20, 40], RANDOM_PORTFOLIOS=2000,
    N_PCA_FACTORS=3,
)
if SMOKE:   # tiny, fast configuration used only to test the code end to end
    CFG.update(QUOTAS=dict(Equity=70, ETF=30, Fund=20, Crypto=25, Index=10, Currency=15),
               MODEL_SEEDS=[42, 7], SENS_SEEDS=[42], HP_BUDGET=2, TASK1_MAX_EPOCHS=6, TASK1_PATIENCE=3,
               TASK2_MAX_EPOCHS=6, TASK2_PATIENCE=3, CKPT_EVERY=2, BOOTSTRAP_REPS=40, PERM_REPEATS=1,
               PORTFOLIO_SIZES=[5], RANDOM_PORTFOLIOS=50, MIN_OVERLAP_DAILY=150, ASSET_MIN_OBS_H0=100)

TASK1_MODELS = ["GCN", "GraphSAGE", "GAT", "R-GCN", "Gated-RelGAT"]
GNN_MODELS = ["GCN", "GraphSAGE", "GAT", "R-GCN", "Gated-RelGAT"] if RUN_PROFILE == "full" else ["GAT", "R-GCN"]
NEURAL_T2 = ["Pair-only"] + GNN_MODELS
RELATIONS = ["semantic_similarity", "same_sector", "same_fund_category",
             "same_country", "same_exchange", "same_currency"]

# Hyperparameter grids. Every learned model gets exactly CFG["HP_BUDGET"] configurations.
NEURAL_GRID = [dict(lr=lr, hid=h, dropout=d) for lr in (1e-3, 3e-3) for h in (32, 64) for d in (0.1, 0.3)]
LOGIT_GRID = [dict(C=c) for c in (0.01, 0.03, 0.1, 0.3, 1.0, 3.0)]
XGB_GRID = [dict(max_depth=md, learning_rate=lr, n_estimators=ne)
            for md, lr, ne in [(3, 0.05, 400), (3, 0.1, 300), (5, 0.05, 400),
                               (5, 0.1, 300), (7, 0.05, 300), (4, 0.08, 350)]]
_hp_rng = np.random.default_rng(2021)
NEURAL_GRID = [NEURAL_GRID[i] for i in sorted(_hp_rng.choice(len(NEURAL_GRID), CFG["HP_BUDGET"], replace=False))]
LOGIT_GRID, XGB_GRID = LOGIT_GRID[:CFG["HP_BUDGET"]], XGB_GRID[:CFG["HP_BUDGET"]]

# ---------- Drive / folders ----------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB and not SMOKE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/KG_Strong_Pipeline")
    LOCAL = Path("/content/kg_local")
else:
    BASE = Path(os.environ.get("KG_BASE", Path.cwd() / "KG_Strong_Pipeline"))
    LOCAL = BASE / "_local"
ROOT = BASE / f"study_{STUDY}"
DIR = {k: ROOT / k for k in ("stages", "runs", "results", "figures", "audits")}
DIR["shared"] = BASE / "shared"
DIR["prices"] = DIR["shared"] / "prices"
for p in list(DIR.values()) + [LOCAL]:
    p.mkdir(parents=True, exist_ok=True)
CFG_SIG = hashlib.sha256(json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:10]
(DIR["audits"] / "config.json").write_text(json.dumps(CFG, indent=2, default=str))
print(f"STUDY {STUDY} | profile {RUN_PROFILE} | smoke {SMOKE} | config signature {CFG_SIG}")
print("Drive root:", ROOT)

Mounted at /content/drive
STUDY B | profile full | smoke False | config signature 3c6cc5290d
Drive root: /content/drive/MyDrive/KG_Strong_Pipeline/study_B


In [ ]:
# CELL 0.3 — checkpoint engine, logging, audits (everything goes to Drive)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def log(msg):
    line = f"{datetime.now():%Y-%m-%d %H:%M:%S} | {msg}"
    print(line)
    with open(DIR["audits"] / "run_log.txt", "a") as fh:
        fh.write(line + "\n")

def _atomic(path, writer):
    """Write to a temp file, then rename: a disconnect can never leave a half-written result."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp{os.getpid()}")
    writer(tmp)
    os.replace(tmp, path)

def save_pkl(obj, path):
    def w(p):
        with open(p, "wb") as fh:
            pickle.dump(obj, fh, protocol=4)
    _atomic(path, w)

def load_pkl(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)

def save_torch(obj, path):
    _atomic(path, lambda p: torch.save(obj, p))

def load_torch(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def save_csv(df, name, folder="results"):
    path = DIR[folder] / f"{name}.csv"
    _atomic(path, lambda p: df.to_csv(p, index=False))
    return path

def sig(*parts):
    return hashlib.sha256(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:12]

def stage(name, fn, deps=()):
    """Run fn() once; afterwards load its saved result from Drive."""
    path = DIR["stages"] / f"{name}__{sig(name, CFG_SIG, deps)}.pkl"
    if path.exists():
        try:
            obj = load_pkl(path)
            log(f"[loaded from Drive] {name}")
            return obj
        except Exception as e:  # corrupted file → recompute
            log(f"[corrupt checkpoint, recomputing] {name}: {e}")
    t0 = time.time()
    obj = fn()
    save_pkl(obj, path)
    log(f"[computed + saved] {name} ({time.time() - t0:.0f}s)")
    return obj

_AUDITS = {}
def audit(name, ok, detail=""):
    _AUDITS[name] = dict(check=name, passed=bool(ok), detail=str(detail), time=f"{datetime.now():%H:%M:%S}")
    save_csv(pd.DataFrame(_AUDITS.values()), "audit_log", folder="audits")
    if not ok:
        raise AssertionError(f"AUDIT FAILED: {name} | {detail}")
    print(f"  ✓ {name}" + (f" | {detail}" if detail else ""))

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "device": DEVICE}
for mod in ("torch_geometric", "sklearn", "xgboost", "sentence_transformers", "yfinance", "leidenalg"):
    try:
        versions[mod] = __import__(mod).__version__
    except Exception:
        versions[mod] = "n/a"
(DIR["audits"] / "versions.json").write_text(json.dumps(versions, indent=2))
set_seed(42)
log(f"setup complete | device {DEVICE} | {versions}")

2026-10-06 12:24:39 | setup complete | device cpu | {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'device': 'cpu', 'torch_geometric': '2.8.0.post1', 'sklearn': '1.6.1', 'xgboost': '3.4.1', 'sentence_transformers': '5.7.0', 'yfinance': '0.2.66', 'leidenalg': '0.12.0'}


## 1 · Point-in-time universe from the historical FinanceDatabase commit
The exact commit is fetched with a depth-1 fetch (fast and reliable). Parsed tables are cached to Drive,
so a lost local disk never forces a re-download.

In [ ]:
# CELL 1.1 — fetch the historical commit and parse every asset class
def fetch_findb():
    if os.environ.get("KG_FDB_PATH"):
        return Path(os.environ["KG_FDB_PATH"])
    repo = LOCAL / "FinanceDatabase_hist"
    if not (repo / "Database").is_dir():
        shutil.rmtree(repo, ignore_errors=True)
        repo.mkdir(parents=True)
        cmds = [["git", "init", "-q"],
                ["git", "remote", "add", "origin", "https://github.com/JerBouma/FinanceDatabase.git"],
                ["git", "fetch", "-q", "--depth", "1", "origin", CFG["FINDB_COMMIT"]],
                ["git", "checkout", "-q", "FETCH_HEAD"]]
        for c in cmds:
            subprocess.run(c, cwd=repo, check=True)
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True, text=True).stdout.strip()
    date = subprocess.run(["git", "log", "-1", "--format=%cI"], cwd=repo, capture_output=True, text=True).stdout.strip()
    audit("historical FinanceDatabase commit pinned", head == CFG["FINDB_COMMIT"], f"{head[:12]} committed {date}")
    audit("commit predates graph snapshot", pd.Timestamp(date[:10]) <= pd.Timestamp(CFG["GRAPH_SNAPSHOT_DATE"]), date[:10])
    return repo

def _jload(p):
    try:
        with open(p, encoding="utf-8") as fh:
            return json.load(fh)
    except UnicodeDecodeError:
        with open(p, encoding="latin-1") as fh:
            return json.load(fh)

def parse_findb():
    db = fetch_findb() / "Database"
    rows = []
    def add(sym, rec, cls, category=None):
        if not sym or not isinstance(rec, dict):
            return
        rows.append(dict(symbol=str(sym).strip(), asset_type=cls,
                         short_name=rec.get("short_name"), long_name=rec.get("long_name"),
                         summary=rec.get("summary"), currency=rec.get("currency"),
                         sector=rec.get("sector") if cls == "Equity" else None,
                         industry=rec.get("industry") if cls == "Equity" else None,
                         fund_category=(rec.get("category") or category) if cls in ("ETF", "Fund") else None,
                         country=rec.get("country") if cls == "Equity" else None,
                         exchange=rec.get("exchange"), family=rec.get("family"),
                         coin=rec.get("cryptocurrency")))
    for part in ("Equities Part 1.json", "Equities Part 2.json"):
        for s, r in _jload(db / "Equities" / part).items():
            add(s, r, "Equity")
    for cls, folder in (("ETF", "ETFs"), ("Fund", "Funds")):
        for p in sorted((db / folder).glob("*.json")):
            if p.name.startswith("_") or p.name.startswith("core_"):
                continue
            for s, r in _jload(p).items():
                add(s, r, cls, category=p.stem)
        agg = db / folder / f"_{folder}.json"          # superset, includes uncategorised products
        if agg.exists():
            for s, r in _jload(agg).items():
                add(s, r, cls)
    for p in sorted((db / "Cryptocurrencies").glob("*.json")):
        if not p.name.startswith("_"):
            for s, r in _jload(p).items():
                add(s, r, "Crypto")
    for p in sorted((db / "Currencies").glob("*.json")):
        if not p.name.startswith("_"):
            for s, r in _jload(p).items():
                add(s, r, "Currency")
    for p in sorted((db / "Indices").glob("*.json")):
        if not p.name.startswith("_"):
            for s, r in _jload(p).items():
                add(s, r, "Index")
    df = pd.DataFrame(rows)
    # categorised file entries come first, so keep="first" retains the category label
    df = df.drop_duplicates(["asset_type", "symbol"], keep="first").reset_index(drop=True)
    return df

RAW_PATH = DIR["shared"] / f"findb_raw_{CFG['FINDB_COMMIT'][:12]}.parquet"
if RAW_PATH.exists():
    RAW = pd.read_parquet(RAW_PATH)
    log(f"[loaded from Drive] parsed FinanceDatabase ({len(RAW):,} rows)")
else:
    RAW = parse_findb()
    _atomic(RAW_PATH, lambda p: RAW.to_parquet(p, index=False))
    log(f"[computed + saved] parsed FinanceDatabase ({len(RAW):,} rows)")
display(RAW.asset_type.value_counts().rename("records in historical commit").to_frame())

2026-10-06 12:24:47 | [loaded from Drive] parsed FinanceDatabase (342,290 rows)


,records in historical commit
asset_type,
Equity,155727
Index,86329
Fund,57918
ETF,36734
Crypto,3724
Currency,1858


In [ ]:
# CELL 1.2 — pre-specified, deterministic universe selection with duplicate-instrument control
INVALID = {"", "unknown", "nan", "none", "null", "n/a", "na"}
def valid(x):
    return x is not None and not (isinstance(x, float) and np.isnan(x)) and str(x).strip().lower() not in INVALID

_PRICE_SENT = re.compile(r"[^.]*\b(price|supply|trading on|market cap|traded|24 hours|volume)\b[^.]*\.", re.I)
def clean_summary(s, cls):
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    if cls == "Crypto":   # remove dated market-data sentences, keep the descriptive text
        s = _PRICE_SENT.sub("", s).strip()
    return s

def norm_name(s):
    s = str(s or "").lower()
    s = re.split(r"\s+-\s+|\bclass\b|\bshares?\b|\bacc\b|\binc\b|\bdist\b", s)[0]
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def stable_rank(sym):
    return int(hashlib.sha256(f"{sym}|{CFG['SPLIT_SEED']}".encode()).hexdigest()[:12], 16)

def select_universe():
    df = RAW.copy()
    df["name"] = np.where(df.long_name.map(valid), df.long_name, df.short_name)
    df = df[df.name.map(valid)].copy()
    df["summary_clean"] = [clean_summary(s, c) for s, c in zip(df.summary, df.asset_type)]
    df["has_summary"] = df.summary_clean.str.len() >= CFG["MIN_SUMMARY_CHARS"]
    report = []
    parts = []
    for cls, quota in CFG["QUOTAS"].items():
        d = df[df.asset_type == cls].copy()
        n0 = len(d)
        if cls == "Equity":
            d = d[d.exchange.isin(CFG["PRIMARY_EQUITY_EXCHANGES"])]
        if cls == "Crypto":       # one USD listing per coin
            d = d[d.symbol.str.endswith("-USD")]
            d["coin_key"] = np.where(d.coin.map(valid), d.coin, d.symbol.str.replace("-USD", ""))
            d = d.drop_duplicates("coin_key")
        if cls == "Currency":     # one USD rate per currency
            d = d[d.symbol.str.endswith("USD=X") & (d.symbol.str.len() == 8)]
        d["rank"] = d.symbol.map(stable_rank)
        d = d.sort_values(["has_summary", "rank"], ascending=[False, True])
        n1 = len(d)
        # identical descriptions or names = share classes / cross-listings of one instrument
        d["sum_key"] = np.where(d.has_summary, d.summary_clean.str.lower().str[:400], d.symbol)
        d = d.drop_duplicates("sum_key")
        if cls in ("Equity", "ETF", "Fund"):
            d["nm_key"] = d.name.map(norm_name)
            d = d[d.nm_key != ""].drop_duplicates("nm_key")
        n2 = len(d)
        take = d.head(quota)
        parts.append(take)
        report.append(dict(asset_class=cls, records=n0, after_listing_rules=n1, after_dedup=n2,
                           quota=quota, selected=len(take), with_description=int(take.has_summary.sum())))
    u = pd.concat(parts, ignore_index=True)
    u["text"] = np.where(u.has_summary, u.name + ". " + u.summary_clean, u.name)
    u["asset_id"] = u.asset_type + "::" + u.symbol
    u = u.sort_values(["asset_type", "symbol"]).reset_index(drop=True)
    u["idx"] = np.arange(len(u))
    keep = ["idx", "asset_id", "symbol", "asset_type", "name", "text", "has_summary", "sector", "industry",
            "fund_category", "country", "exchange", "currency"]
    return u[keep], pd.DataFrame(report)

ASSETS, SELECTION_REPORT = stage("universe", select_universe)
N = len(ASSETS)
save_csv(SELECTION_REPORT, "universe_selection")
display(SELECTION_REPORT)
audit("asset identifiers unique", ASSETS.asset_id.is_unique, f"{N} assets")
audit("one listing per crypto coin", ASSETS[ASSETS.asset_type == "Crypto"].symbol.str.endswith("-USD").all())
audit("descriptions de-duplicated within class",
      not ASSETS[ASSETS.has_summary].duplicated(["asset_type", "text"]).any())
for c, q in CFG["QUOTAS"].items():
    got = int((ASSETS.asset_type == c).sum())
    if got < q:
        log(f"note: {c} quota {q} capped at {got} available instruments after duplicate control")

2026-10-06 12:24:48 | [loaded from Drive] universe


,asset_class,records,after_listing_rules,after_dedup,quota,selected,with_description
0,Equity,112497,31311,25559,3000,3000,3000
1,ETF,36725,36725,13950,1200,1200,1200
2,Fund,56497,56497,25883,800,800,800
3,Crypto,3724,371,371,600,371,370
4,Index,75506,75506,75506,400,400,0
5,Currency,1858,134,134,300,134,0


  ✓ asset identifiers unique | 5905 assets
  ✓ one listing per crypto coin
  ✓ descriptions de-duplicated within class
2026-10-06 12:24:49 | note: Crypto quota 600 capped at 371 available instruments after duplicate control
2026-10-06 12:24:49 | note: Currency quota 300 capped at 134 available instruments after duplicate control


## 2 · Frozen MPNet embeddings and the six-relation graph

In [ ]:
# CELL 2.1 — frozen sentence embeddings (chunked, resumable)
def embed_texts():
    texts = ASSETS.text.tolist()
    if SMOKE:   # no model download in the smoke test: deterministic topic-structured vectors
        rng = np.random.default_rng(0)
        topics = {t: rng.normal(size=768) for t in ASSETS.asset_type.unique()}
        X = np.stack([topics[t] + 0.8 * np.random.default_rng(stable_rank(s) % 2**32).normal(size=768)
                      for t, s in zip(ASSETS.asset_type, ASSETS.symbol)])
        return (X / np.linalg.norm(X, axis=1, keepdims=True)).astype(np.float32)
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(CFG["TEXT_MODEL"], device=DEVICE)
    chunk_dir = DIR["stages"] / f"emb_chunks_{sig(texts)}"
    chunk_dir.mkdir(exist_ok=True)
    out = []
    for k in range(0, len(texts), 1024):
        f = chunk_dir / f"chunk_{k:06d}.npy"
        if not f.exists():
            e = model.encode(texts[k:k + 1024], batch_size=64 if DEVICE == "cuda" else 16,
                             normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
            _atomic(f, lambda p, e=e: np.save(open(p, "wb"), e.astype(np.float32)))
            log(f"embedded {min(k + 1024, len(texts))}/{len(texts)}")
        out.append(np.load(f))
    return np.vstack(out).astype(np.float32)

X_TEXT = stage("embeddings", embed_texts, deps=[ASSETS.asset_id.tolist()])
TYPES = sorted(ASSETS.asset_type.unique())
X_TYPE = np.stack([(ASSETS.asset_type == t).to_numpy(np.float32) for t in TYPES], 1)
X_NODE = np.concatenate([X_TEXT, X_TYPE, ASSETS.has_summary.to_numpy(np.float32)[:, None]], 1)
audit("embeddings finite and unit-norm", np.isfinite(X_TEXT).all() and np.allclose(np.linalg.norm(X_TEXT, axis=1), 1, atol=1e-3))
audit("no relation-defining metadata in node features", X_NODE.shape[1] == 768 + len(TYPES) + 1,
      f"node dim {X_NODE.shape[1]} = text 768 + class one-hot {len(TYPES)} + description flag")

2026-10-06 12:24:50 | [loaded from Drive] embeddings
  ✓ embeddings finite and unit-norm
  ✓ no relation-defining metadata in node features | node dim 775 = text 768 + class one-hot 6 + description flag


In [ ]:
# CELL 2.2 — six historical relation families (no return information)
from sklearn.neighbors import NearestNeighbors

def rowdot(X, i, j, chunk=200_000):
    """(X[i] * X[j]).sum(1) in chunks, so millions of pairs never materialise two (pairs × 768) arrays."""
    i, j = np.asarray(i), np.asarray(j)
    out = np.empty(len(i), np.float32)
    for a in range(0, len(i), chunk):
        out[a:a + chunk] = np.einsum("ij,ij->i", X[i[a:a + chunk]], X[j[a:a + chunk]])
    return out

def canon(E):
    E = np.asarray(E, dtype=np.int64).reshape(-1, 2)
    E = E[E[:, 0] != E[:, 1]]
    if len(E) == 0:
        return np.zeros((0, 2), np.int64)
    return np.unique(np.sort(E, 1), axis=0)

def attr_edges(labels, rel):
    labels = np.asarray(labels, dtype=object)
    out = []
    for g in pd.unique(labels):
        if not valid(g):
            continue
        m = np.where(labels == g)[0]
        if len(m) < 2:
            continue
        for i in m:
            others = m[m != i]
            if len(others) > CFG["ATTR_CAP"]:
                r = np.random.default_rng(int(hashlib.sha256(f"{rel}|{i}".encode()).hexdigest()[:8], 16))
                others = r.choice(others, CFG["ATTR_CAP"], replace=False)
            out.extend((int(i), int(j)) for j in others)
    return canon(out)

def build_graph():
    nn_ = NearestNeighbors(n_neighbors=CFG["K_SEM"] + 1, metric="cosine").fit(X_TEXT)
    _, ind = nn_.kneighbors(X_TEXT)
    sem = canon([(i, int(j)) for i in range(N) for j in ind[i, 1:]])
    return {"semantic_similarity": sem,
            "same_sector": attr_edges(ASSETS.sector.values, "sector"),
            "same_fund_category": attr_edges(ASSETS.fund_category.values, "fund_category"),
            "same_country": attr_edges(ASSETS.country.values, "country"),
            "same_exchange": attr_edges(ASSETS.exchange.values, "exchange"),
            "same_currency": attr_edges(ASSETS.currency.values, "currency")}

REL = stage("graph", build_graph, deps=[ASSETS.asset_id.tolist()])
UNION = canon(np.vstack([E for E in REL.values() if len(E)]))
atype = ASSETS.asset_type.to_numpy()
graph_tab = pd.DataFrame([dict(relation=r, edges=len(E), cross_asset_edges=int((atype[E[:, 0]] != atype[E[:, 1]]).sum()) if len(E) else 0)
                          for r, E in REL.items()])
graph_tab["cross_asset_pct"] = 100 * graph_tab.cross_asset_edges / graph_tab.edges.replace(0, np.nan)
import networkx as nx
_G = nx.Graph(); _G.add_nodes_from(range(N)); _G.add_edges_from(map(tuple, UNION))
graph_union = pd.DataFrame([dict(nodes=N, union_edges=len(UNION), components=nx.number_connected_components(_G),
                                 relation_edge_instances=int(graph_tab.edges.sum()))])
save_csv(graph_tab, "graph_relations"); save_csv(graph_union, "graph_union")
display(graph_tab); display(graph_union)
audit("graph built without returns", True, "edges from text kNN and snapshot metadata only")

2026-10-06 12:24:51 | [loaded from Drive] graph


,relation,edges,cross_asset_edges,cross_asset_pct
0,semantic_similarity,66294,5986,9.029475
1,same_sector,67169,0,0.000000
2,same_fund_category,6273,2427,38.689622
3,same_country,64520,0,0.000000
4,same_exchange,134913,27303,20.237486
5,same_currency,140208,80991,57.764892


,nodes,union_edges,components,relation_edge_instances
0,5905,432138,1,479377


  ✓ graph built without returns | edges from text kNN and snapshot metadata only


## 3 · Prices and returns
Prices are downloaded in small batches into Drive shards with a per-symbol manifest; a restart skips
every symbol already attempted. Returns are computed on a **common Monday–Friday calendar** so that
crypto (7-day) and exchange-traded assets measure returns over the same horizon.

In [ ]:
# CELL 3.1 — resumable price download (shared by both studies)
def yahoo_symbol(row):
    s = row.symbol
    return s if row.asset_type != "Index" or s.startswith("^") else "^" + s

ASSETS["yahoo"] = ASSETS.apply(yahoo_symbol, axis=1)
MANIFEST = DIR["prices"] / "manifest.csv"

def read_manifest():
    return pd.read_csv(MANIFEST) if MANIFEST.exists() else pd.DataFrame(columns=["yahoo", "status", "n_obs", "attempts"])

def synthetic_prices(symbols):
    """Smoke test only: factor-structured prices with persistent co-movement."""
    dates = pd.bdate_range(CFG["DOWNLOAD_START"], CFG["DOWNLOAD_END"])
    rng = np.random.default_rng(1)
    T = len(dates)
    mkt = rng.normal(0, 0.01, T)
    cls_f = {t: rng.normal(0, 0.01, T) for t in TYPES}
    grp_f = {g: rng.normal(0, 0.012, T) for g in range(12)}
    out = {}
    lookup = ASSETS.set_index("yahoo")
    for s in symbols:
        r = lookup.loc[s]
        g = stable_rank(str(r.exchange) + str(r.sector) + str(r.fund_category)) % 12
        z = np.random.default_rng(stable_rank(s) % 2**32)
        b = z.uniform(0.3, 1.2, 3)
        ret = b[0] * mkt + b[1] * cls_f[r.asset_type] + b[2] * grp_f[g] + z.normal(0, 0.012, T)
        p = 100 * np.exp(np.cumsum(ret))
        if z.random() < 0.1:
            p[: z.integers(100, 400)] = np.nan
        out[s] = p
    return pd.DataFrame(out, index=dates)

def _is_rate_limit(msg):
    m = str(msg).lower()
    return "ratelimit" in m.replace(" ", "") or "rate limited" in m or "too many requests" in m

def download_prices(symbols=None):
    """Resumable download. Yahoo's own error log separates 'rate-limited' (retry later) from
    'no data / delisted' (genuinely unavailable, tried twice and then left out)."""
    import yfinance as yf
    man = read_manifest()
    done = set(man.yahoo[(man.status == "ok") | ((man.status == "empty") & (man.attempts >= 2)) | (man.attempts >= 5)])
    universe = list(pd.unique(ASSETS.yahoo if symbols is None else pd.Series(symbols)))
    todo = [s for s in universe if s not in done]
    log(f"prices: {len(done)} symbols settled, {len(todo)} to (re)try")
    batch, streak = 40, 0
    for k in range(0, len(todo), batch):
        syms = todo[k:k + batch]
        shard = DIR["prices"] / f"shard_{sig(syms)}.parquet"
        close, errs = {}, {}
        try:
            try:
                yf.shared._ERRORS.clear()
            except Exception:
                pass
            raw = yf.download(syms, start=CFG["DOWNLOAD_START"], end=CFG["DOWNLOAD_END"], auto_adjust=True,
                              progress=False, threads=True, group_by="ticker")
            errs = dict(getattr(yf.shared, "_ERRORS", {}) or {})
            for s in syms:
                try:
                    c = raw[s]["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw["Close"]
                    c = c.dropna()
                    if len(c):
                        close[s] = c
                except Exception:
                    pass
            if close:
                dfc = pd.DataFrame(close)
                dfc.index = pd.to_datetime(dfc.index).tz_localize(None)
                _atomic(shard, lambda p, d=dfc: d.to_parquet(p))
            rows = [dict(yahoo=s, status="ok" if s in close else ("ratelimited" if _is_rate_limit(errs.get(s, "")) else "empty"),
                         n_obs=len(close.get(s, []))) for s in syms]
        except Exception as e:
            rows = [dict(yahoo=s, status="ratelimited" if _is_rate_limit(e) else "error", n_obs=0) for s in syms]
        man = read_manifest()
        prev_att = man.set_index("yahoo").attempts.to_dict() if len(man) else {}
        for r in rows:   # a rate-limited request does not count as an attempt
            r["attempts"] = int(prev_att.get(r["yahoo"], 0)) + (0 if r["status"] == "ratelimited" else 1)
        man = pd.concat([man[~man.yahoo.isin(syms)], pd.DataFrame(rows)], ignore_index=True)
        _atomic(MANIFEST, lambda p, m=man: m.to_csv(p, index=False))
        if (k // batch) % 10 == 0:
            log(f"prices: {min(k + batch, len(todo))}/{len(todo)} tried this session")
        if sum(r["status"] == "ratelimited" for r in rows) >= len(rows) / 2:
            streak += 1
            log(f"Yahoo is rate-limiting (batch {streak} in a row); pausing 120 s")
            if streak >= 4:
                break
            time.sleep(120)
        else:
            streak = 0
            time.sleep(1.0)
    man = read_manifest()
    pending = [s for s in universe
               if s not in set(man.yahoo) or man.set_index("yahoo").status.get(s) in ("ratelimited", "error")]
    share = len(pending) / max(len(universe), 1)
    if pending and share <= 0.05:
        log(f"WARNING: {len(pending)} symbols ({100 * share:.1f}%) still rate-limited; continuing without them. "
            "A later Run all retries them, and Task 2 is recomputed automatically if any arrive.")
    elif pending:
        raise RuntimeError(f"Yahoo rate limit: {len(pending)} symbols still pending. Everything downloaded so far is "
                           "saved on Drive. Wait 30–60 minutes (or Runtime → Disconnect and delete runtime, then reconnect) "
                           "and Run all again — the download continues where it stopped.")
    log(f"prices settled: {man.status.value_counts().to_dict()}")

def load_price_panel():
    shards = sorted(DIR["prices"].glob("shard_*.parquet"))
    frames = [pd.read_parquet(s) for s in shards]
    if not frames:
        return pd.DataFrame()
    P = pd.concat(frames, axis=1)
    P = P.loc[:, ~P.columns.duplicated(keep="last")]
    return P.sort_index()

if SMOKE:
    PRICES = synthetic_prices(ASSETS.yahoo.tolist())
else:
    download_prices()
    PRICES = load_price_panel()
cov = ASSETS.assign(priced=ASSETS.yahoo.isin(PRICES.columns)).groupby("asset_type").priced.agg(["size", "sum"])
cov.columns = ["assets", "with_prices"]
cov["coverage_pct"] = 100 * cov.with_prices / cov.assets
save_csv(cov.reset_index(), "price_coverage")
display(cov)

2026-10-06 12:25:00 | prices: 6287 symbols settled, 0 to (re)try
2026-10-06 12:25:29 | prices settled: {'ok': 4183, 'empty': 2322}


,assets,with_prices,coverage_pct
asset_type,,,
Crypto,371,205,55.256065
Currency,134,115,85.820896
ETF,1200,926,77.166667
Equity,3000,1984,66.133333
Fund,800,569,71.125000
Index,400,2,0.500000


In [ ]:
# CELL 3.2 — returns on a common business-day calendar, with data-quality filters
def make_returns(P, freq="D", yahoo=None, cols=None, crypto=None):
    yahoo = ASSETS.yahoo if yahoo is None else pd.Index(yahoo)
    P = P.reindex(columns=yahoo).copy()
    P.columns = ASSETS.idx.values if cols is None else cols
    if freq == "D":
        cal = pd.bdate_range(P.index.min(), P.index.max())
        Pd = P.reindex(cal)                      # price only on days the asset actually traded
        R = np.log(Pd).diff()                     # NaN unless both consecutive business days traded
    else:
        Pw = P.ffill(limit=4).resample("W-FRI").last()
        R = np.log(Pw).diff()
    R = R.replace([np.inf, -np.inf], np.nan)
    crypto = (ASSETS.asset_type == "Crypto").to_numpy() if crypto is None else np.asarray(crypto)
    lim = np.where(crypto, 1.0, 0.5) if freq == "D" else np.where(crypto, 1.5, 0.8)
    bad = R.abs() > lim
    R = R.mask(bad)
    Z = (R.to_numpy() == 0)
    run = np.zeros(Z.shape, dtype=np.int32)
    for t in range(1, len(Z)):                    # length of the current run of zero returns
        run[t] = (run[t - 1] + 1) * Z[t]
    R = R.mask(run >= 10)                         # ≥10 identical closes in a row: stale quotes
    return R, int(bad.sum().sum())

R_DAILY, n_bad_d = stage("returns_daily", lambda: make_returns(PRICES, "D"), deps=[sig(PRICES.shape, list(PRICES.columns))])
R_WEEKLY, n_bad_w = stage("returns_weekly", lambda: make_returns(PRICES, "W"), deps=[sig(PRICES.shape, list(PRICES.columns))])
log(f"daily returns {R_DAILY.shape}, {n_bad_d} extreme values removed; weekly {R_WEEKLY.shape}")
audit("returns use business-day calendar", R_DAILY.index.dayofweek.max() <= 4)

def window(R, w):
    a, b = CFG["WINDOWS"][w]
    return R.loc[(R.index >= a) & (R.index < b)]

def pairwise_corr(Rw):
    """Pairwise-complete Pearson correlation and overlap counts, fully vectorised."""
    X = Rw.to_numpy(np.float64)
    M = np.isfinite(X).astype(np.float64)
    Z = np.where(M > 0, X, 0.0)
    n = M.T @ M
    sx, sy = Z.T @ M, M.T @ Z
    sxx, syy = (Z ** 2).T @ M, M.T @ (Z ** 2)
    sxy = Z.T @ Z
    num = n * sxy - sx * sy
    den = np.sqrt(np.clip(n * sxx - sx ** 2, 0, None) * np.clip(n * syy - sy ** 2, 0, None))
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.where(den > 0, num / den, np.nan)
    return np.clip(C, -1, 1), n

h0_obs = window(R_DAILY, "H0").notna().sum().to_numpy()
ASSETS["h0_obs"] = h0_obs
DATA_SIG = sig(R_DAILY.shape, int(R_DAILY.notna().sum().sum()), int(R_WEEKLY.notna().sum().sum()))[:6]
man_ = read_manifest() if not SMOKE else pd.DataFrame(columns=["status"])
log(f"return-data signature {DATA_SIG}; price manifest status: {man_.status.value_counts().to_dict()}")

2026-10-06 12:25:39 | [loaded from Drive] returns_daily
2026-10-06 12:25:40 | [loaded from Drive] returns_weekly
2026-10-06 12:25:40 | daily returns (1333, 5905), 1772 extreme values removed; weekly (268, 5905)
  ✓ returns use business-day calendar
2026-10-06 12:25:40 | return-data signature 2d66b2; price manifest status: {'ok': 4183, 'empty': 2322}


## 4 · Models and a resumable trainer
One trainer serves both tasks. Each run is identified by a key; a finished run is stored as
`runs/<key>.final.pt` (scores + best weights) and a running one as `runs/<key>.partial.pt`, written every
`CKPT_EVERY` epochs. A restart resumes from the partial checkpoint with the optimiser state intact.

In [ ]:
# CELL 4.1 — encoders, symmetric decoder, metrics
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, RGCNConv
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, f1_score,
                             matthews_corrcoef, precision_score, recall_score, balanced_accuracy_score)

class HomoEnc(nn.Module):
    def __init__(self, kind, din, hid, out, dropout, heads=4):
        super().__init__()
        self.kind, self.dp = kind, dropout
        if kind == "GCN":
            self.c1, self.c2 = GCNConv(din, hid), GCNConv(hid, out)
        elif kind == "GraphSAGE":
            self.c1, self.c2 = SAGEConv(din, hid), SAGEConv(hid, out)
        else:  # GAT: heads × (hid/heads) then a single-head output layer
            self.c1 = GATConv(din, max(hid // heads, 4), heads=heads, dropout=dropout)
            self.c2 = GATConv(max(hid // heads, 4) * heads, out, heads=1, dropout=dropout)
        h1 = max(hid // heads, 4) * heads if kind == "GAT" else hid
        self.n1, self.n2 = nn.LayerNorm(h1), nn.LayerNorm(out)
        self.skip = nn.Linear(din, out, bias=False)
    def forward(self, x, g):
        h = F.dropout(F.elu(self.n1(self.c1(x, g["homo"]))), self.dp, self.training)
        return self.n2(self.c2(h, g["homo"]) + self.skip(x))

class RGCNEnc(nn.Module):
    def __init__(self, din, hid, out, dropout, n_rel):
        super().__init__()
        self.dp = dropout
        self.c1, self.c2 = RGCNConv(din, hid, n_rel, num_bases=min(4, n_rel)), RGCNConv(hid, out, n_rel, num_bases=min(4, n_rel))
        self.n1, self.n2 = nn.LayerNorm(hid), nn.LayerNorm(out)
        self.skip = nn.Linear(din, out, bias=False)
    def forward(self, x, g):
        ei, et = g["rgcn"]
        h = F.dropout(F.elu(self.n1(self.c1(x, ei, et))), self.dp, self.training)
        return self.n2(self.c2(h, ei, et) + self.skip(x))

class GatedRelGAT(nn.Module):
    """One attention stream per relation, mixed by learned node-wise gates."""
    def __init__(self, din, hid, out, dropout, rels):
        super().__init__()
        self.rels, self.dp = rels, dropout
        self.l1 = nn.ModuleDict({r: GATConv(din, hid, heads=1, dropout=dropout, add_self_loops=True) for r in rels})
        self.l2 = nn.ModuleDict({r: GATConv(hid, out, heads=1, dropout=dropout, add_self_loops=True) for r in rels})
        self.g1, self.g2 = nn.Linear(din, len(rels)), nn.Linear(hid, len(rels))
        self.n1, self.n2 = nn.LayerNorm(hid), nn.LayerNorm(out)
        self.skip = nn.Linear(din, out, bias=False)
    def _mix(self, x, layers, gate, g):
        H = torch.stack([layers[r](x, g["rel"][r]) for r in self.rels], 1)
        a = torch.softmax(gate(x), 1).unsqueeze(-1)
        return (a * H).sum(1)
    def forward(self, x, g):
        h = F.dropout(F.elu(self.n1(self._mix(x, self.l1, self.g1, g))), self.dp, self.training)
        return self.n2(self._mix(h, self.l2, self.g2, g) + self.skip(x))

def make_encoder(kind, din, hid, dropout):
    out = 64
    if kind in ("GCN", "GraphSAGE", "GAT"):
        return HomoEnc(kind, din, hid, out, dropout)
    if kind == "R-GCN":
        return RGCNEnc(din, hid, out, dropout, len(RELATIONS))
    if kind == "Gated-RelGAT":
        return GatedRelGAT(din, hid, out, dropout, RELATIONS)
    return None   # Pair-only

class LinkModel(nn.Module):
    """Symmetric decoder on [z_i ⊙ z_j, |z_i − z_j|, p_ij]; with no encoder it is the pair-only model."""
    def __init__(self, kind, din, pair_dim, hid, dropout):
        super().__init__()
        self.enc = make_encoder(kind, din, hid, dropout)
        dz = 128 if self.enc is not None else 0
        self.dec = nn.Sequential(nn.Linear(dz + pair_dim, hid), nn.LayerNorm(hid), nn.ELU(),
                                 nn.Dropout(dropout), nn.Linear(hid, 1))
    def embed(self, x, g):
        return self.enc(x, g) if self.enc is not None else None
    def score(self, z, pairs, pf):
        parts = []
        if z is not None:
            zi, zj = z[pairs[:, 0]], z[pairs[:, 1]]
            parts += [zi * zj, (zi - zj).abs()]
        if pf is not None:
            parts.append(pf)
        return self.dec(torch.cat(parts, 1)).squeeze(-1)

def to_ei(E, n_off=None):
    E = np.asarray(E, dtype=np.int64).reshape(-1, 2)
    if len(E) == 0:
        return torch.zeros((2, 0), dtype=torch.long)
    return torch.tensor(np.vstack([E, E[:, ::-1]]).T.copy(), dtype=torch.long)

def make_graph(rel_edges):
    """rel_edges: dict relation → (m,2) local edges. Returns tensors for every encoder type."""
    homo = canon(np.vstack([E for E in rel_edges.values() if len(E)])) if any(len(E) for E in rel_edges.values()) else np.zeros((0, 2), np.int64)
    eis, ets = [], []
    for k, r in enumerate(RELATIONS):
        ei = to_ei(rel_edges.get(r, np.zeros((0, 2))))
        eis.append(ei); ets.append(torch.full((ei.shape[1],), k, dtype=torch.long))
    g = {"homo": to_ei(homo), "rel": {r: to_ei(rel_edges.get(r, np.zeros((0, 2)))) for r in RELATIONS},
         "rgcn": (torch.cat(eis, 1), torch.cat(ets))}
    return g

def graph_to(g, dev):
    return {"homo": g["homo"].to(dev), "rel": {r: e.to(dev) for r, e in g["rel"].items()},
            "rgcn": (g["rgcn"][0].to(dev), g["rgcn"][1].to(dev))}

def best_f1_threshold(y, s):
    p, r, t = precision_recall_curve(y, s)
    if len(t) == 0:
        return 0.5
    f = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    return float(t[int(np.nanargmax(f))])

def metrics(y, s, thr):
    y = np.asarray(y).astype(int); s = np.asarray(s, float)
    pred = (s >= thr).astype(int)
    k = max(int(y.sum()), 1)
    top = np.argsort(-s)
    return dict(AUC=roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan,
                AP=average_precision_score(y, s) if y.sum() > 0 else np.nan,
                F1=f1_score(y, pred, zero_division=0),
                MCC=matthews_corrcoef(y, pred) if len(np.unique(pred)) > 1 else 0.0,
                Precision=precision_score(y, pred, zero_division=0), Recall=recall_score(y, pred, zero_division=0),
                BalancedAcc=balanced_accuracy_score(y, pred), R_Precision=float(y[top[:k]].mean()),
                P_at_100=float(y[top[:100]].mean()), Flagged=int(pred.sum()), N=len(y), Positives=int(y.sum()),
                Prevalence=float(y.mean()), Threshold=float(thr))

In [ ]:
# CELL 4.2 — resumable trainer (epoch-level checkpoints on Drive)
def summarise(df, by, metrics_=("AUC", "AP", "F1", "MCC")):
    from scipy import stats
    rows = []
    for k, g in df.groupby(by):
        r = dict(zip(by, k if isinstance(k, tuple) else (k,)))
        r["runs"] = len(g)
        for m in metrics_:
            v = g[m].dropna().to_numpy()
            r[f"{m}_mean"] = v.mean() if len(v) else np.nan
            r[f"{m}_sd"] = v.std(ddof=1) if len(v) > 1 else np.nan
            if len(v) > 1:
                h = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
                r[f"{m}_ci_l"], r[f"{m}_ci_u"] = v.mean() - h, v.mean() + h
            r[f"{m}_min"], r[f"{m}_max"] = (v.min(), v.max()) if len(v) else (np.nan, np.nan)
        rows.append(r)
    return pd.DataFrame(rows)

def run_path(key, kind):
    return DIR["runs"] / f"{key}.{kind}.pt"

def train_link(key, kind, hp, seed, data, max_epochs, patience):
    """data: dict with x, graphs (train/val/test), pairs, pair features, labels per split.
    Returns dict(result row, val_scores, test_scores, extra_scores)."""
    fin = run_path(key, "final")
    if fin.exists():
        try:
            return load_torch(fin)
        except Exception as e:
            log(f"[corrupt final, retraining] {key}: {e}")
    set_seed(seed)
    dev = DEVICE
    x = {s: torch.tensor(data["x"][s], dtype=torch.float32, device=dev) for s in ("train", "val", "test")}
    g = {s: graph_to(data["graph"][s], dev) for s in ("train", "val", "test")}
    P = {s: torch.tensor(data["pairs"][s], dtype=torch.long, device=dev) for s in ("train", "val", "test")}
    PF = {s: (torch.tensor(data["pf"][s], dtype=torch.float32, device=dev) if data["pf"] is not None else None)
          for s in ("train", "val", "test")}
    y_tr = torch.tensor(data["y"]["train"], dtype=torch.float32, device=dev)
    pdim = data["pf"]["train"].shape[1] if data["pf"] is not None else 0
    model = LinkModel(kind, data["x"]["train"].shape[1], pdim, hp["hid"], hp["dropout"]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=CFG["WEIGHT_DECAY"])
    pos = float(y_tr.sum().item()); neg = float(len(y_tr) - pos)
    pos_w = torch.tensor(neg / max(pos, 1.0), device=dev)
    state = dict(epoch=0, best=-1.0, best_state=None, best_epoch=0, wait=0, history=[])
    part = run_path(key, "partial")
    if part.exists():
        try:
            ck = load_torch(part)
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); state = ck["state"]
            torch.set_rng_state(ck["rng"])
            log(f"[resume] {key} from epoch {state['epoch']}")
        except Exception as e:
            log(f"[partial unreadable, restarting] {key}: {e}")

    def predict(split):
        model.eval()
        with torch.no_grad():
            z = model.embed(x[split], g[split])
            out = []
            for b in range(0, len(P[split]), CFG["PAIR_BATCH"]):
                pf = PF[split][b:b + CFG["PAIR_BATCH"]] if PF[split] is not None else None
                out.append(torch.sigmoid(model.score(z, P[split][b:b + CFG["PAIR_BATCH"]], pf)).cpu())
        return torch.cat(out).numpy()

    n_tr = len(P["train"])
    while state["epoch"] < max_epochs and state["wait"] < patience:
        model.train()
        perm = torch.randperm(n_tr, device=dev)
        tot = 0.0
        for b in range(0, n_tr, CFG["PAIR_BATCH"]):
            idx = perm[b:b + CFG["PAIR_BATCH"]]
            opt.zero_grad()
            z = model.embed(x["train"], g["train"])
            pf = PF["train"][idx] if PF["train"] is not None else None
            logit = model.score(z, P["train"][idx], pf)
            loss = F.binary_cross_entropy_with_logits(logit, y_tr[idx], pos_weight=pos_w)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
            opt.step()
            tot += loss.item() * len(idx)
        vs = predict("val")
        vap = average_precision_score(data["y"]["val"], vs)
        state["epoch"] += 1
        state["history"].append(dict(epoch=state["epoch"], loss=tot / n_tr, val_AP=vap))
        if vap > state["best"] + 1e-6:
            state.update(best=vap, best_epoch=state["epoch"], wait=0,
                         best_state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        else:
            state["wait"] += 1
        if state["epoch"] % CFG["CKPT_EVERY"] == 0:
            save_torch(dict(model=model.state_dict(), opt=opt.state_dict(), state=state, rng=torch.get_rng_state()), part)
    model.load_state_dict(state["best_state"])
    vs, ts = predict("val"), predict("test")
    extra = {}
    for name in data.get("extra", {}):   # additional scoring sets (e.g. portfolio formation pairs)
        ex = data["extra"][name]
        xe = torch.tensor(ex["x"], dtype=torch.float32, device=dev); ge = graph_to(ex["graph"], dev)
        pe = torch.tensor(ex["pairs"], dtype=torch.long, device=dev)
        fe = torch.tensor(ex["pf"], dtype=torch.float32, device=dev) if ex["pf"] is not None else None
        model.eval()
        with torch.no_grad():
            z = model.embed(xe, ge)
            extra[name] = np.concatenate([torch.sigmoid(model.score(z, pe[b:b + CFG["PAIR_BATCH"]],
                                          fe[b:b + CFG["PAIR_BATCH"]] if fe is not None else None)).cpu().numpy()
                                          for b in range(0, len(pe), CFG["PAIR_BATCH"])]) if len(pe) else np.zeros(0)
    thr = best_f1_threshold(data["y"]["val"], vs)
    row = dict(key=key, model=kind, seed=seed, best_epoch=state["best_epoch"], val_AP=state["best"], **hp,
               **metrics(data["y"]["test"], ts, thr))
    out = dict(row=row, val_scores=vs.astype(np.float32), test_scores=ts.astype(np.float32), extra=extra,
               best_state=state["best_state"], history=state["history"], hp=hp)
    save_torch(out, fin)
    if part.exists():
        part.unlink()
    log(f"[run done] {key} | test AP {row['AP']:.4f} | epoch {state['best_epoch']}")
    return out

In [ ]:
# CELL 5.L — Task 1 results (computed by the main notebook; loaded here, not recomputed)
import gc
_t1 = DIR["results"] / "task1_runs.csv"
if not _t1.exists():
    raise FileNotFoundError(f"{_t1} not found. Run the main notebook (STUDY = '{STUDY}') through section 5 first.")
_t1df = pd.read_csv(_t1)
_base = _t1df.model.isin(["Text cosine", "Logistic regression"])
T1_BASE, T1_RUNS = _t1df[_base].reset_index(drop=True), _t1df[~_base].reset_index(drop=True)
log(f"Task 1 results loaded: {len(T1_RUNS)} GNN runs, {len(T1_BASE)} baselines")
gc.collect()

2026-10-06 12:25:41 | Task 1 results loaded: 50 GNN runs, 2 baselines


30

## 6 · Task 2 — prospective co-movement: data
* Eligibility: ≥120 valid daily returns in H0 (2020-08 → 2021-08), for **every** asset class.
* Node-disjoint 60/20/20 split stratified by class; each split's message-passing graph is induced on its
  own nodes only.
* Two rolling folds: **F1** train t1 / validate t2 / test t3 (2023–24); **F2** train t2 / validate t3 /
  test t4 (2024–25). The graph is always the 2021 snapshot, so both test years are fully prospective.
* Target: Pearson correlation in the target window > τ with ≥180 overlapping days (35 weeks weekly).
  All evaluable pairs are kept — natural prevalence, no negative sampling.
* Lagged correlation comes from the window immediately before each target window; the multi-factor
  residual version removes market, asset-class and three principal-component factors estimated inside
  that (past) window.

In [ ]:
# CELL 6.1 — eligibility and node-disjoint split
def node_split():
    elig = ASSETS.idx[ASSETS.h0_obs >= CFG["ASSET_MIN_OBS_H0"]].to_numpy()
    rng = np.random.default_rng(CFG["SPLIT_SEED"])
    out = {"train": [], "val": [], "test": []}
    for t in TYPES:
        m = elig[atype[elig] == t]
        m = m[rng.permutation(len(m))]
        a = int(round(CFG["SPLIT_FRACTIONS"][0] * len(m))); b = a + int(round(CFG["SPLIT_FRACTIONS"][1] * len(m)))
        out["train"] += m[:a].tolist(); out["val"] += m[a:b].tolist(); out["test"] += m[b:].tolist()
    return {k: np.array(sorted(v), dtype=np.int64) for k, v in out.items()}

SPLIT = stage("task2_node_split", node_split, deps=[ASSETS.h0_obs.tolist()])
audit("node splits mutually exclusive",
      not (set(SPLIT["train"]) & set(SPLIT["val"]) or set(SPLIT["train"]) & set(SPLIT["test"]) or set(SPLIT["val"]) & set(SPLIT["test"])))
audit("eligibility uses H0 only", True, f"{sum(len(v) for v in SPLIT.values())} eligible assets with ≥{CFG['ASSET_MIN_OBS_H0']} H0 returns")
elig_tab = (ASSETS.assign(eligible=ASSETS.h0_obs >= CFG["ASSET_MIN_OBS_H0"]).groupby("asset_type").eligible.agg(["size", "sum"])
            .rename(columns={"size": "assets", "sum": "eligible"}))
for s in SPLIT:
    elig_tab[s] = pd.Series(atype[SPLIT[s]]).value_counts()
save_csv(elig_tab.fillna(0).reset_index(), "task2_eligibility"); display(elig_tab.fillna(0))

2026-10-06 12:25:42 | [loaded from Drive] task2_node_split
  ✓ node splits mutually exclusive
  ✓ eligibility uses H0 only | 3685 eligible assets with ≥120 H0 returns


,assets,eligible,train,val,test
asset_type,,,,,
Crypto,371,203,122,41.0,40
Currency,134,112,67,22.0,23
ETF,1200,883,530,177.0,176
Equity,3000,1925,1155,385.0,385
Fund,800,560,336,112.0,112
Index,400,2,1,0.0,1


In [ ]:
# CELL 6.2 — candidate pairs, lagged and factor-residual lagged correlations
REL_KEYS = {r: set((E[:, 0] * N + E[:, 1]).tolist()) for r, E in REL.items()}
TYPE_PAIRS = list(itertools.combinations_with_replacement(TYPES, 2))
TP_INDEX = {tp: k for k, tp in enumerate(TYPE_PAIRS)}

def residualise(Rw, train_cols):
    """Returns minus fitted market, asset-class and PCA factors (all estimated inside the window)."""
    X = Rw.to_numpy(float)
    mkt = np.nanmean(X, 1)
    facs = [mkt]
    for t in TYPES:
        cols = np.where(atype[Rw.columns.to_numpy()] == t)[0]
        if len(cols) >= 5:
            facs.append(np.nanmean(X[:, cols], 1))
    tr = np.isin(Rw.columns.to_numpy(), train_cols)
    Z = X[:, tr]
    Z = (Z - np.nanmean(Z, 0)) / (np.nanstd(Z, 0) + 1e-12)
    Z = np.nan_to_num(Z)
    if Z.shape[1] > CFG["N_PCA_FACTORS"]:
        U, S, _ = np.linalg.svd(Z, full_matrices=False)
        facs += [U[:, k] * S[k] for k in range(CFG["N_PCA_FACTORS"])]
    Fm = np.column_stack([np.nan_to_num(f) for f in facs] + [np.ones(len(X))])
    out = np.full_like(X, np.nan)
    for j in range(X.shape[1]):
        ok = np.isfinite(X[:, j])
        if ok.sum() > Fm.shape[1] + 10:
            beta, *_ = np.linalg.lstsq(Fm[ok], X[ok, j], rcond=None)
            out[ok, j] = X[ok, j] - Fm[ok] @ beta
    return pd.DataFrame(out, index=Rw.index, columns=Rw.columns)

def pair_block(nodes, target_w, R, min_overlap, require_target=True, lag_min=None, lag_w=None):
    """All within-set pairs with target correlation (if evaluable), lagged raw and residual correlations."""
    nodes = np.asarray(nodes)
    lag_w = lag_w or CFG["PREV"][target_w]
    Ct, nt = pairwise_corr(window(R, target_w)[nodes])
    Rl = window(R, lag_w)[nodes]
    Cl, nl = pairwise_corr(Rl)
    Cr, _ = pairwise_corr(residualise(Rl, SPLIT["train"]))
    iu, ju = np.triu_indices(len(nodes), 1)
    keep = nt[iu, ju] >= min_overlap if require_target else np.ones(len(iu), bool)
    iu, ju = iu[keep], ju[keep]
    lag_ok = nl[iu, ju] >= (lag_min or CFG["LAG_MIN_OVERLAP"])
    return dict(i=nodes[iu], j=nodes[ju], li=iu, lj=ju, lag_n=nl[iu, ju],
                target=np.where(nt[iu, ju] >= min_overlap, Ct[iu, ju], np.nan),
                lag=np.where(lag_ok, Cl[iu, ju], np.nan), lag_res=np.where(lag_ok, Cr[iu, ju], np.nan))

def build_fold(fold, freq):
    R = R_DAILY if freq == "D" else R_WEEKLY
    mo = CFG["MIN_OVERLAP_DAILY"] if freq == "D" else CFG["MIN_OVERLAP_WEEKLY"]
    lm = CFG["LAG_MIN_OVERLAP"] if freq == "D" else CFG["LAG_MIN_OVERLAP_WEEKLY"]
    blocks = {s: pair_block(SPLIT[s], fold[s], R, mo, lag_min=lm) for s in ("train", "val", "test")}
    # portfolio formation set: every pair of test assets with enough history before the test window
    hist = window(R_DAILY, CFG["PREV"][fold["test"]]).notna().sum()
    form_nodes = SPLIT["test"][hist.reindex(SPLIT["test"]).to_numpy() >= CFG["ASSET_MIN_OBS_H0"]]
    blocks["formation"] = pair_block(form_nodes, fold["test"], R_DAILY, mo, require_target=False)
    blocks["formation"]["nodes"] = form_nodes
    return blocks

FOLD_DATA = {}
for f in CFG["FOLDS"]:
    for freq in ("D", "W"):
        if freq == "W" and f["name"] != "F1":
            continue
        FOLD_DATA[(f["name"], freq)] = stage(f"task2_pairs_{f['name']}_{freq}", lambda f=f, freq=freq: build_fold(f, freq),
                                             deps=[SPLIT["train"].tolist(), DATA_SIG])

rows = []
for (fn, freq), B in FOLD_DATA.items():
    for s in ("train", "val", "test"):
        y = B[s]["target"] > CFG["TAU_MAIN"]
        cross = atype[B[s]["i"]] != atype[B[s]["j"]]
        rows.append(dict(fold=fn, freq=freq, split=s, nodes=len(SPLIT[s]), candidates=len(y), positives=int(y.sum()),
                         prevalence_pct=100 * y.mean(), cross_asset_candidates=int(cross.sum()),
                         cross_asset_positives=int((y & cross).sum())))
T2_SAMPLE = pd.DataFrame(rows)
save_csv(T2_SAMPLE, "task2_sample"); display(T2_SAMPLE)
for (fn, freq), B in FOLD_DATA.items():
    for s in ("train", "val", "test"):
        audit(f"{fn}/{freq} {s} pairs within one split", np.isin(B[s]["i"], SPLIT[s]).all() and np.isin(B[s]["j"], SPLIT[s]).all())
audit("target windows start after graph snapshot",
      all(pd.Timestamp(CFG["WINDOWS"][f[s]][0]) > pd.Timestamp(CFG["GRAPH_SNAPSHOT_DATE"]) for f in CFG["FOLDS"] for s in ("train", "val", "test")))
audit("lag windows end where target windows begin",
      all(CFG["WINDOWS"][CFG["PREV"][w]][1] == CFG["WINDOWS"][w][0] for w in CFG["PREV"]))

2026-10-06 12:25:46 | [loaded from Drive] task2_pairs_F1_D
2026-10-06 12:25:52 | [loaded from Drive] task2_pairs_F1_W
2026-10-06 12:25:56 | [loaded from Drive] task2_pairs_F2_D


,fold,freq,split,nodes,candidates,positives,prevalence_pct,cross_asset_candidates,cross_asset_positives
0,F1,D,train,2211,2367218,140506,5.935491,1528626,83814
1,F1,D,val,737,258053,15288,5.924364,166885,8843
2,F1,D,test,737,261630,11397,4.356152,168490,6464
3,F1,W,train,2211,2432076,215508,8.861072,1564271,134350
4,F1,W,val,737,266786,28954,10.852893,171971,18250
5,F1,W,test,737,268269,27024,10.073471,172891,16001
6,F2,D,train,2211,2346410,132051,5.627789,1513986,79217
7,F2,D,val,737,255660,10886,4.257999,165394,5719
8,F2,D,test,737,262085,13854,5.286071,169429,8212


  ✓ F1/D train pairs within one split
  ✓ F1/D val pairs within one split
  ✓ F1/D test pairs within one split
  ✓ F1/W train pairs within one split
  ✓ F1/W val pairs within one split
  ✓ F1/W test pairs within one split
  ✓ F2/D train pairs within one split
  ✓ F2/D val pairs within one split
  ✓ F2/D test pairs within one split
  ✓ target windows start after graph snapshot
  ✓ lag windows end where target windows begin


In [ ]:
# CELL 6.3 — pair features (fitted on training pairs only) and split-induced graphs
def pair_features(B, lagkey, stats=None):
    i, j = B["i"], B["j"]
    lo, hi = np.minimum(i, j), np.maximum(i, j)
    key = lo * N + hi
    lag = B[lagkey]
    miss = ~np.isfinite(lag)
    if stats is None:
        stats = {"median": float(np.nanmedian(lag)) if np.isfinite(lag).any() else 0.0}
    lag_f = np.where(miss, stats["median"], lag)
    cos = rowdot(X_TEXT, i, j)
    tp = np.zeros((len(i), len(TYPE_PAIRS)), np.float32)
    tp[np.arange(len(i)), [TP_INDEX[tuple(sorted((a, b)))] for a, b in zip(atype[i], atype[j])]] = 1
    rel = np.column_stack([np.isin(key, list(REL_KEYS[r])) for r in RELATIONS]).astype(np.float32)
    Xf = np.column_stack([cos, lag_f, miss, atype[i] == atype[j], rel, tp]).astype(np.float32)
    if "mu" not in stats:
        stats["mu"], stats["sd"] = Xf.mean(0), Xf.std(0) + 1e-6
    return (Xf - stats["mu"]) / stats["sd"], stats

FEATURE_GROUPS = {"Description cosine": [0], "Lagged correlation (+ missing flag)": [1, 2], "Same asset class": [3],
                  "Relation indicators": list(range(4, 4 + len(RELATIONS))),
                  "Asset-class-pair indicators": list(range(4 + len(RELATIONS), 4 + len(RELATIONS) + len(TYPE_PAIRS)))}

def induced(nodes, rel_edges=None):
    rel_edges = REL if rel_edges is None else rel_edges
    pos = {int(n): k for k, n in enumerate(nodes)}
    out = {}
    for r, E in rel_edges.items():
        m = np.isin(E[:, 0], nodes) & np.isin(E[:, 1], nodes)
        out[r] = np.array([[pos[a], pos[b]] for a, b in E[m]], dtype=np.int64).reshape(-1, 2)
    return out

GRAPHS = {s: make_graph(induced(SPLIT[s])) for s in ("train", "val", "test")}
used = set(np.unique(np.concatenate([GRAPHS["train"]["homo"].numpy().ravel(), np.arange(0)]))) if GRAPHS["train"]["homo"].numel() else set()
audit("no validation/test node in the training message-passing graph",
      max(used, default=-1) < len(SPLIT["train"]), "training graph indexed on training nodes only")

def task2_inputs(fold_name, freq, tau, lagkey):
    B = FOLD_DATA[(fold_name, freq)]
    pf_tr, st = pair_features(B["train"], lagkey)
    data = dict(x={}, graph={}, pairs={}, pf={}, y={}, cross={}, extra={})
    for s in ("train", "val", "test"):
        pf = pf_tr if s == "train" else pair_features(B[s], lagkey, st)[0]
        data["x"][s] = X_NODE[SPLIT[s]]
        data["graph"][s] = GRAPHS[s]
        data["pairs"][s] = np.column_stack([B[s]["li"], B[s]["lj"]])
        data["pf"][s] = pf
        data["y"][s] = (B[s]["target"] > tau).astype(np.float32)
        data["cross"][s] = atype[B[s]["i"]] != atype[B[s]["j"]]
    Fm = B["formation"]
    pos = {int(n): k for k, n in enumerate(SPLIT["test"])}
    data["extra"]["formation"] = dict(x=X_NODE[SPLIT["test"]], graph=GRAPHS["test"],
                                      pairs=np.column_stack([[pos[a] for a in Fm["i"]], [pos[b] for b in Fm["j"]]]).astype(np.int64).reshape(-1, 2),
                                      pf=pair_features(Fm, lagkey, st)[0])
    data["raw"] = {s: dict(cos=rowdot(X_TEXT, B[s]["i"], B[s]["j"]), lag=B[s]["lag"], lag_res=B[s]["lag_res"])
                   for s in ("train", "val", "test", "formation")}
    data["stats"] = st
    return data

  ✓ no validation/test node in the training message-passing graph | training graph indexed on training nodes only


## 7 · Task 2 — models, equal-budget tuning, multi-seed runs
Every learned model (logistic, XGBoost, pair-only network and each GNN) receives the same number of
hyperparameter configurations, chosen on **F1 validation AP with seed 42 only**. The locked test years
never choose a model, epoch, threshold or configuration.

In [ ]:
# CELL 7.1 — non-neural models and a cached runner for every model type
from xgboost import XGBClassifier

def cross_ap(y, s, cross):
    return average_precision_score(y[cross], s[cross]) if y[cross].sum() > 0 else np.nan

def run_model(setting, model, hp, seed, data):
    """setting: (fold, freq, tau, lagkey). Returns dict(row, val_scores, test_scores, formation_scores)."""
    fold, freq, tau, lagkey = setting
    hpid = sig(hp)[:6] if hp else "none"
    key = f"T2_{DATA_SIG}_{fold}_{freq}_tau{tau:.2f}_{lagkey}_{model}_{hpid}_s{seed}"
    if model in NEURAL_T2:
        out = train_link(key, model, hp, seed, data, CFG["TASK2_MAX_EPOCHS"], CFG["TASK2_PATIENCE"])
        res = dict(row=dict(out["row"]), val=out["val_scores"], test=out["test_scores"], formation=out["extra"].get("formation"))
    else:
        fin = run_path(key, "final")
        res = load_torch(fin) if fin.exists() else None
    if model not in NEURAL_T2 and res is None:
        set_seed(seed)
        raw = data["raw"]
        if model == "Text cosine":
            sc = {s: raw[s]["cos"] for s in raw}
        elif model in ("Lagged correlation", "Residual lagged correlation"):
            k = "lag" if model == "Lagged correlation" else "lag_res"
            med = np.nanmedian(raw["train"][k]) if np.isfinite(raw["train"][k]).any() else 0.0
            sc = {s: np.where(np.isfinite(raw[s][k]), raw[s][k], med) for s in raw}
        else:
            if model == "Logistic":
                clf = LogisticRegression(C=hp["C"], max_iter=3000, class_weight="balanced")
            else:
                pw = (len(data["y"]["train"]) - data["y"]["train"].sum()) / max(data["y"]["train"].sum(), 1)
                clf = XGBClassifier(**hp, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                    scale_pos_weight=pw, random_state=seed, n_jobs=-1, eval_metric="aucpr",
                                    device="cuda" if DEVICE == "cuda" else "cpu")
            clf.fit(data["pf"]["train"], data["y"]["train"])
            sc = {s: clf.predict_proba(data["pf"][s])[:, 1] for s in ("val", "test")}
            fpf = data.get("extra", {}).get("formation", {}).get("pf")
            sc["formation"] = clf.predict_proba(fpf)[:, 1] if fpf is not None and len(fpf) else np.zeros(0)
        thr = best_f1_threshold(data["y"]["val"], sc["val"])
        row = dict(key=key, model=model, seed=seed, **(hp or {}), **metrics(data["y"]["test"], sc["test"], thr))
        res = dict(row=row, val=np.asarray(sc["val"], np.float32), test=np.asarray(sc["test"], np.float32),
                   formation=np.asarray(sc.get("formation", np.zeros(0)), np.float32))
        save_torch(res, fin)
    res["row"].update(fold=fold, freq=freq, tau=tau, lag=lagkey,
                      cross_AP=cross_ap(data["y"]["test"], res["test"], data["cross"]["test"]),
                      cross_positives=int(data["y"]["test"][data["cross"]["test"]].sum()))
    return res

DATA_CACHE = {}
def get_data(setting):
    if setting not in DATA_CACHE:
        DATA_CACHE.clear()          # keep memory bounded: one setting at a time
        DATA_CACHE[setting] = task2_inputs(*setting)
    return DATA_CACHE[setting]

In [ ]:
# missing imports (were in the skipped Task-1 cell)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
# CELL 7.2 — equal-budget hyperparameter selection (F1 validation, seed 42)
GRIDS = {"Logistic": LOGIT_GRID, "XGBoost": XGB_GRID, **{m: NEURAL_GRID for m in NEURAL_T2}}
HP_SETTING = ("F1", "D", CFG["TAU_MAIN"], "lag")

def select_hps():
    d = get_data(HP_SETTING)
    rows = []
    for m, grid in GRIDS.items():
        for hp in grid:
            r = run_model(HP_SETTING, m, hp, 42, d)
            rows.append(dict(model=m, hp=json.dumps(hp), val_AP=average_precision_score(d["y"]["val"], r["val"])))
    tab = pd.DataFrame(rows)
    best = tab.loc[tab.groupby("model").val_AP.idxmax()].reset_index(drop=True)
    return tab, best

HP_TAB, HP_BEST = select_hps()
save_csv(HP_TAB, "task2_hp_search"); save_csv(HP_BEST, "task2_hp_selected")
BEST_HP = {r.model: json.loads(r.hp) for r in HP_BEST.itertuples()}
display(HP_BEST)
audit("hyperparameters selected on validation only", True, f"budget {CFG['HP_BUDGET']} per learned model")

,model,hp,val_AP
0,GAT,"{""lr"": 0.003, ""hid"": 64, ""dropout"": 0.3}",0.827719
1,GCN,"{""lr"": 0.003, ""hid"": 64, ""dropout"": 0.1}",0.833210
2,Gated-RelGAT,"{""lr"": 0.003, ""hid"": 64, ""dropout"": 0.3}",0.829714
3,GraphSAGE,"{""lr"": 0.003, ""hid"": 64, ""dropout"": 0.1}",0.824209
4,Logistic,"{""C"": 0.01}",0.865034
5,Pair-only,"{""lr"": 0.001, ""hid"": 64, ""dropout"": 0.3}",0.868695
6,R-GCN,"{""lr"": 0.003, ""hid"": 64, ""dropout"": 0.1}",0.839045
7,XGBoost,"{""max_depth"": 5, ""learning_rate"": 0.05, ""n_est...",0.862699


  ✓ hyperparameters selected on validation only | budget 6 per learned model


In [ ]:
# CELL 7.3 — main runs: both folds, τ = 0.5, ten seeds per stochastic model
DET = ["Text cosine", "Lagged correlation", "Residual lagged correlation"]
SCORES = {}      # (setting, model) → {seed: result dict}
ALL_ROWS = []

def run_suite(setting, models, seeds):
    d = get_data(setting)
    for m in models:
        ss = [42] if m in DET else seeds
        for s in ss:
            r = run_model(setting, m, BEST_HP.get(m), s, d)
            SCORES.setdefault((setting, m), {})[s] = r       # keyed by seed: re-running never duplicates
            ALL_ROWS.append(r["row"])

for f in CFG["FOLDS"]:
    run_suite((f["name"], "D", CFG["TAU_MAIN"], "lag"), DET + ["Logistic", "XGBoost"] + NEURAL_T2, CFG["MODEL_SEEDS"])
    log(f"main suite finished for {f['name']}")

2026-10-06 12:28:06 | main suite finished for F1
2026-10-06 12:29:49 | main suite finished for F2


In [ ]:
# CELL 7.4 — sensitivity: threshold τ, weekly frequency, multi-factor residual lag
SENS_MODELS = ["Text cosine", "Lagged correlation", "Logistic", "XGBoost", "Pair-only", "GAT"]
for tau in CFG["TAU_SENS"]:
    run_suite(("F1", "D", tau, "lag"), SENS_MODELS, CFG["SENS_SEEDS"])
run_suite(("F1", "W", CFG["TAU_MAIN"], "lag"), SENS_MODELS, CFG["SENS_SEEDS"])
for f in CFG["FOLDS"]:
    run_suite((f["name"], "D", CFG["TAU_MAIN"], "lag_res"), ["Logistic", "Pair-only"], CFG["SENS_SEEDS"])
T2_RUNS = pd.DataFrame(ALL_ROWS).drop_duplicates("key")
save_csv(T2_RUNS, "task2_runs")
T2_SUM = summarise(T2_RUNS, ["fold", "freq", "tau", "lag", "model"], ("AUC", "AP", "F1", "MCC", "cross_AP", "R_Precision", "P_at_100"))
save_csv(T2_SUM, "task2_summary")
main = T2_SUM[(T2_SUM.freq == "D") & (T2_SUM.tau == CFG["TAU_MAIN"]) & (T2_SUM.lag == "lag")]
display(main.pivot_table(index="model", columns="fold", values=["AP_mean", "cross_AP_mean"]).round(4))

AP_mean         cross_AP_mean        
fold                             F1      F2            F1      F2
model                                                            
GAT                          0.8272  0.7327        0.7991  0.7185
GCN                          0.8196  0.7303        0.7922  0.7129
Gated-RelGAT                 0.8134  0.7323        0.7948  0.7184
GraphSAGE                    0.8207  0.7349        0.7958  0.7184
Lagged correlation           0.8633  0.7406        0.8421  0.7135
Logistic                     0.8662  0.7183        0.8465  0.6871
Pair-only                    0.8478  0.7230        0.8362  0.6963
R-GCN                        0.8147  0.7315        0.7823  0.7092
Residual lagged correlation  0.4033  0.3040        0.3746  0.2705
Text cosine                  0.2555  0.2482        0.2652  0.2203
XGBoost                      0.8644  0.7235        0.8424  0.6976

## 8 · Inference: node-cluster bootstrap for every key contrast
Pairs that share an asset are dependent, so the bootstrap resamples **test assets** with replacement; a
pair (i, j) receives weight c_i·c_j where c is the number of times each asset is drawn. AP is computed
exactly (ties handled as in scikit-learn) under these weights, averaged over seeds, and differenced.
Holm adjustment is applied across contrasts within each fold.

In [ ]:
# CELL 8.1 — weighted AP and the bootstrap
class FastAP:
    """Pre-sorts one score vector so weighted AP costs O(n) per bootstrap replicate."""
    def __init__(self, y, s):
        self.o = np.argsort(-s, kind="mergesort")
        self.y = y[self.o].astype(float)
        ss = s[self.o]
        self.last = np.r_[np.nonzero(np.diff(ss))[0], len(ss) - 1]   # end of each tie group
    def __call__(self, w):
        w = w[self.o]
        tp = np.cumsum(w * self.y)[self.last]
        fp = np.cumsum(w * (1 - self.y))[self.last]
        P = tp[-1]
        if P <= 0:
            return np.nan
        prec = tp / np.maximum(tp + fp, 1e-12)
        rec = tp / P
        return float(np.sum(np.diff(np.r_[0, rec]) * prec))

def bootstrap_contrasts(setting, contrasts, reps):
    d = get_data(setting)
    y = d["y"]["test"]; li, lj = d["pairs"]["test"][:, 0], d["pairs"]["test"][:, 1]
    n_nodes = len(SPLIT["test"])
    models = sorted({m for c in contrasts for m in c})
    fast = {m: [FastAP(y, r["test"]) for r in SCORES[(setting, m)].values()] for m in models}
    point = {m: np.mean([average_precision_score(y, r["test"]) for r in SCORES[(setting, m)].values()]) for m in models}
    rng = np.random.default_rng(12345)
    draws = {c: np.empty(reps) for c in contrasts}
    for b in range(reps):
        cnt = np.bincount(rng.integers(0, n_nodes, n_nodes), minlength=n_nodes).astype(float)
        w = cnt[li] * cnt[lj]
        ap = {m: np.nanmean([f(w) for f in fast[m]]) for m in models}
        for c in contrasts:
            draws[c][b] = ap[c[0]] - ap[c[1]]
    rows = []
    for c in contrasts:
        dd = draws[c][np.isfinite(draws[c])]
        p = 2 * min((dd <= 0).mean(), (dd >= 0).mean())
        rows.append(dict(fold=setting[0], freq=setting[1], tau=setting[2], lag=setting[3], model=c[0], versus=c[1],
                         AP_model=point[c[0]], AP_versus=point[c[1]], diff=point[c[0]] - point[c[1]],
                         boot_se=dd.std(ddof=1), ci_l=np.percentile(dd, 2.5), ci_u=np.percentile(dd, 97.5),
                         p_boot=min(1.0, max(p, 1 / len(dd))), reps=len(dd)))
    out = pd.DataFrame(rows)
    order = np.argsort(out.p_boot.to_numpy())
    m_ = len(out); adj = np.empty(m_); run = 0
    for k, idx in enumerate(order):
        run = max(run, min(1.0, (m_ - k) * out.p_boot.iloc[idx])); adj[idx] = run
    out["p_holm"] = adj
    return out

CONTRASTS = ([("Logistic", "Lagged correlation"), ("XGBoost", "Lagged correlation"), ("Pair-only", "Lagged correlation"),
              ("Lagged correlation", "Text cosine"), ("Pair-only", "Logistic")]
             + [(g, "Pair-only") for g in GNN_MODELS])
BOOT = pd.concat([stage(f"bootstrap_{f['name']}", lambda f=f: bootstrap_contrasts((f["name"], "D", CFG["TAU_MAIN"], "lag"), CONTRASTS, CFG["BOOTSTRAP_REPS"]),
                        deps=[sorted(T2_RUNS.key.tolist())]) for f in CFG["FOLDS"]], ignore_index=True)
save_csv(BOOT, "task2_bootstrap_contrasts")
display(BOOT[["fold", "model", "versus", "diff", "ci_l", "ci_u", "p_holm"]].round(4))

# paired-by-seed contrast (initialisation variability), reported alongside the bootstrap
from scipy import stats as sst
paired = []
for f in CFG["FOLDS"]:
    st_ = (f["name"], "D", CFG["TAU_MAIN"], "lag")
    base = {s: r["row"]["AP"] for s, r in SCORES[(st_, "Pair-only")].items()}
    for g in GNN_MODELS:
        dd = np.array([SCORES[(st_, g)][s]["row"]["AP"] - base[s] for s in base if s in SCORES[(st_, g)]])
        if len(dd) > 1:
            h = sst.t.ppf(0.975, len(dd) - 1) * dd.std(ddof=1) / np.sqrt(len(dd))
            paired.append(dict(fold=f["name"], model=g, versus="Pair-only", seeds=len(dd), diff=dd.mean(), sd=dd.std(ddof=1),
                               ci_l=dd.mean() - h, ci_u=dd.mean() + h, p_t=sst.ttest_1samp(dd, 0).pvalue))
PAIRED = pd.DataFrame(paired); save_csv(PAIRED, "task2_paired_seed_contrasts"); display(PAIRED.round(4))

2026-10-06 12:32:23 | [loaded from Drive] bootstrap_F1
2026-10-06 12:32:23 | [loaded from Drive] bootstrap_F2


,fold,model,versus,diff,ci_l,ci_u,p_holm
0,F1,Logistic,Lagged correlation,0.0029,-0.0032,0.0094,0.696
1,F1,XGBoost,Lagged correlation,0.0011,-0.0048,0.0070,0.710
2,F1,Pair-only,Lagged correlation,-0.0155,-0.0285,-0.0031,0.024
3,F1,Lagged correlation,Text cosine,0.6078,0.5588,0.6554,0.010
4,F1,Pair-only,Logistic,-0.0184,-0.0275,-0.0094,0.010
5,F1,GCN,Pair-only,-0.0282,-0.0450,-0.0130,0.010
6,F1,GraphSAGE,Pair-only,-0.0272,-0.0424,-0.0136,0.010
7,F1,GAT,Pair-only,-0.0206,-0.0347,-0.0075,0.010
8,F1,R-GCN,Pair-only,-0.0331,-0.0492,-0.0180,0.010
9,F1,Gated-RelGAT,Pair-only,-0.0344,-0.0516,-0.0184,0.010


,fold,model,versus,seeds,diff,sd,ci_l,ci_u,p_t
0,F1,GCN,Pair-only,10,-0.0282,0.0141,-0.0383,-0.0181,0.0001
1,F1,GraphSAGE,Pair-only,10,-0.0272,0.0243,-0.0445,-0.0098,0.0063
2,F1,GAT,Pair-only,10,-0.0206,0.0120,-0.0292,-0.0120,0.0004
3,F1,R-GCN,Pair-only,10,-0.0331,0.0138,-0.0430,-0.0232,0.0000
4,F1,Gated-RelGAT,Pair-only,10,-0.0344,0.0126,-0.0434,-0.0253,0.0000
5,F2,GCN,Pair-only,10,0.0074,0.0058,0.0032,0.0115,0.0031
6,F2,GraphSAGE,Pair-only,10,0.0119,0.0032,0.0097,0.0142,0.0000
7,F2,GAT,Pair-only,10,0.0098,0.0075,0.0044,0.0151,0.0025
8,F2,R-GCN,Pair-only,10,0.0086,0.0099,0.0015,0.0157,0.0232
9,F2,Gated-RelGAT,Pair-only,10,0.0093,0.0082,0.0035,0.0152,0.0056


## 9 · Predictive attribution (frozen models, no retraining, no threshold retuning)

In [ ]:
# CELL 9.1 — grouped pair-feature permutation and relation-channel leave-one-out
def rescore(setting, model, seed, pf_test=None, graph_test=None):
    d = get_data(setting)
    r = load_torch(run_path(SCORES[(setting, model)][seed]["row"]["key"], "final"))
    hp = r["hp"]
    m = LinkModel(model, d["x"]["test"].shape[1], d["pf"]["test"].shape[1], hp["hid"], hp["dropout"]).to(DEVICE)
    m.load_state_dict(r["best_state"]); m.eval()
    x = torch.tensor(d["x"]["test"], dtype=torch.float32, device=DEVICE)
    g = graph_to(graph_test if graph_test is not None else d["graph"]["test"], DEVICE)
    P = torch.tensor(d["pairs"]["test"], dtype=torch.long, device=DEVICE)
    pf = torch.tensor(pf_test if pf_test is not None else d["pf"]["test"], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        z = m.embed(x, g)
        return np.concatenate([torch.sigmoid(m.score(z, P[b:b + CFG["PAIR_BATCH"]], pf[b:b + CFG["PAIR_BATCH"]])).cpu().numpy()
                               for b in range(0, len(P), CFG["PAIR_BATCH"])])

def attribution(fold):
    st_ = (fold, "D", CFG["TAU_MAIN"], "lag")
    d = get_data(st_)
    y, cross = d["y"]["test"], d["cross"]["test"]
    perm_rows, loo_rows = [], []
    for model in [m for m in ("Pair-only", "GAT", "R-GCN") if m in NEURAL_T2]:
        for seed in SCORES[(st_, model)]:
            base = SCORES[(st_, model)][seed]["test"]
            ap0, cap0 = average_precision_score(y, base), cross_ap(y, base, cross)
            rng = np.random.default_rng(seed)
            for gname, cols in FEATURE_GROUPS.items():
                for k in range(CFG["PERM_REPEATS"]):
                    pf = d["pf"]["test"].copy()
                    pf[:, cols] = pf[rng.permutation(len(pf))][:, cols]
                    s = rescore(st_, model, seed, pf_test=pf)
                    perm_rows.append(dict(fold=fold, model=model, seed=seed, group=gname, repeat=k,
                                          dAP=ap0 - average_precision_score(y, s), dAP_cross=cap0 - cross_ap(y, s, cross)))
            if model != "Pair-only":
                ind = induced(SPLIT["test"])
                for r in RELATIONS:
                    gr = make_graph({k: (v if k != r else np.zeros((0, 2), np.int64)) for k, v in ind.items()})
                    s = rescore(st_, model, seed, graph_test=gr)
                    loo_rows.append(dict(fold=fold, model=model, seed=seed, relation=r, test_edges=len(ind[r]),
                                         dAP=ap0 - average_precision_score(y, s), dAP_cross=cap0 - cross_ap(y, s, cross)))
    return pd.DataFrame(perm_rows), pd.DataFrame(loo_rows)

PERM, LOO = [], []
for f in CFG["FOLDS"]:
    p_, l_ = stage(f"attribution_{f['name']}", lambda f=f: attribution(f["name"]), deps=[sorted(T2_RUNS.key.tolist())])
    PERM.append(p_); LOO.append(l_)
PERM, LOO = pd.concat(PERM), pd.concat(LOO)
perm_sum = summarise(PERM.groupby(["fold", "model", "seed", "group"], as_index=False)[["dAP", "dAP_cross"]].mean().rename(columns={"dAP": "AP", "dAP_cross": "cross_AP"}),
                     ["fold", "model", "group"], ("AP", "cross_AP"))
loo_sum = summarise(LOO.rename(columns={"dAP": "AP", "dAP_cross": "cross_AP"}), ["fold", "model", "relation"], ("AP", "cross_AP"))
save_csv(PERM, "attribution_permutation_runs"); save_csv(perm_sum, "attribution_permutation")
save_csv(LOO, "attribution_relation_loo_runs"); save_csv(loo_sum, "attribution_relation_loo")
display(perm_sum[["fold", "model", "group", "AP_mean", "AP_ci_l", "AP_ci_u"]].round(4))

2026-10-06 12:32:24 | [loaded from Drive] attribution_F1
2026-10-06 12:32:24 | [loaded from Drive] attribution_F2


,fold,model,group,AP_mean,AP_ci_l,AP_ci_u
0,F1,GAT,Asset-class-pair indicators,0.1265,0.1110,0.1420
1,F1,GAT,Description cosine,0.0268,0.0148,0.0388
2,F1,GAT,Lagged correlation (+ missing flag),0.6705,0.6578,0.6833
3,F1,GAT,Relation indicators,0.0077,-0.0000,0.0155
4,F1,GAT,Same asset class,0.0018,0.0003,0.0034
5,F1,Pair-only,Asset-class-pair indicators,0.0727,0.0591,0.0864
6,F1,Pair-only,Description cosine,0.0246,0.0179,0.0313
7,F1,Pair-only,Lagged correlation (+ missing flag),0.7653,0.7619,0.7687
8,F1,Pair-only,Relation indicators,0.0056,0.0024,0.0088
9,F1,Pair-only,Same asset class,0.0069,0.0028,0.0110


## 10 · Economic test: does low predicted co-movement buy real diversification?
At the start of each test year, using only information available then, greedily build a K-asset portfolio
of **test-split assets** that minimises the sum of predicted co-movement among its members. Hold it equally
weighted for the test year (missing returns earn zero, i.e. cash) and measure realised volatility and the
realised average pairwise correlation. Benchmarks: 2,000 random portfolios and 2,000 random portfolios
with the **same asset-class mix** (isolating within-class selection skill). One formation per year, so
transaction costs are identical across strategies and omitted.

In [ ]:
# CELL 10.1 — portfolio construction and evaluation
def greedy(S, K):
    n = len(S)
    S = np.where(np.isfinite(S), S, np.nanmedian(S))
    np.fill_diagonal(S, 0)
    chosen = [int(np.argmin(S.sum(1)))]
    while len(chosen) < min(K, n):
        cost = S[:, chosen].sum(1); cost[chosen] = np.inf
        chosen.append(int(np.argmin(cost)))
    return np.array(chosen)

def realised(nodes, Rt):
    X = Rt[nodes].to_numpy(float)
    port = np.nan_to_num(X).mean(1)
    C, n = pairwise_corr(Rt[nodes])
    iu = np.triu_indices(len(nodes), 1)
    ok = n[iu] >= 20
    return float(port.std(ddof=1) * np.sqrt(252)), float(np.nanmean(C[iu][ok])) if ok.any() else np.nan

def economic(fold):
    st_ = (fold["name"], "D", CFG["TAU_MAIN"], "lag")
    d = get_data(st_)
    Fm = FOLD_DATA[(fold["name"], "D")]["formation"]
    nodes = Fm["nodes"]; pos = {int(v): k for k, v in enumerate(nodes)}
    a, b = np.array([pos[i] for i in Fm["i"]]), np.array([pos[j] for j in Fm["j"]])
    Rt = window(R_DAILY, fold["test"])
    def matrix(v):
        S = np.full((len(nodes), len(nodes)), np.nan); S[a, b] = v; S[b, a] = v; return S
    scorers = {"Lagged correlation": matrix(d["raw"]["formation"]["lag"]), "Text cosine": matrix(d["raw"]["formation"]["cos"])}
    for m in ("Logistic", "XGBoost", "Pair-only", "GAT"):
        if (st_, m) in SCORES:
            scorers[m] = matrix(np.mean([r["formation"] for r in SCORES[(st_, m)].values()], 0))
    rng = np.random.default_rng(7)
    types_ = atype[nodes]
    rows = []
    for K in CFG["PORTFOLIO_SIZES"]:
        if K >= len(nodes):
            continue
        rand = np.array([realised(nodes[rng.choice(len(nodes), K, replace=False)], Rt) for _ in range(CFG["RANDOM_PORTFOLIOS"])])
        for name, S in scorers.items():
            ch = greedy(S, K)
            vol, rho = realised(nodes[ch], Rt)
            mix = pd.Series(types_[ch]).value_counts()
            cm = []
            for _ in range(CFG["RANDOM_PORTFOLIOS"]):
                pick = np.concatenate([rng.choice(np.where(types_ == t)[0], c, replace=False) for t, c in mix.items()])
                cm.append(realised(nodes[pick], Rt))
            cm = np.array(cm)
            rows.append(dict(fold=fold["name"], K=K, selector=name, vol=vol, mean_pair_corr=rho,
                             random_vol_mean=np.nanmean(rand[:, 0]), vol_pctile_vs_random=float((rand[:, 0] < vol).mean()),
                             random_corr_mean=np.nanmean(rand[:, 1]), corr_pctile_vs_random=float((rand[:, 1] < rho).mean()),
                             classmatched_vol_mean=np.nanmean(cm[:, 0]), vol_pctile_vs_classmatched=float((cm[:, 0] < vol).mean()),
                             classmatched_corr_mean=np.nanmean(cm[:, 1]), corr_pctile_vs_classmatched=float((cm[:, 1] < rho).mean()),
                             class_mix=json.dumps(mix.to_dict()), formation_assets=len(nodes)))
    return pd.DataFrame(rows)

ECON = pd.concat([stage(f"economic_{f['name']}", lambda f=f: economic(f), deps=[sorted(T2_RUNS.key.tolist())]) for f in CFG["FOLDS"]], ignore_index=True)
save_csv(ECON, "economic_portfolios")
display(ECON[["fold", "K", "selector", "vol", "mean_pair_corr", "vol_pctile_vs_classmatched", "corr_pctile_vs_classmatched"]].round(3))
audit("portfolio formation uses pre-test information only", True,
      "assets screened on the window before the test year; scores from frozen models and lagged features")

2026-10-06 12:32:24 | [loaded from Drive] economic_F1
2026-10-06 12:32:25 | [loaded from Drive] economic_F2


,fold,K,selector,vol,mean_pair_corr,vol_pctile_vs_classmatched,corr_pctile_vs_classmatched
0,F1,20,Lagged correlation,0.078,-0.026,0.211,0.000
1,F1,20,Text cosine,0.131,-0.000,0.728,0.001
2,F1,20,Logistic,0.143,0.007,0.656,0.004
3,F1,20,XGBoost,0.138,0.007,0.490,0.006
4,F1,20,Pair-only,0.172,0.012,0.932,0.032
5,F1,20,GAT,0.274,0.006,0.994,0.004
6,F1,40,Lagged correlation,0.147,-0.004,0.960,0.000
7,F1,40,Text cosine,0.105,0.021,0.780,0.004
8,F1,40,Logistic,0.112,0.009,0.738,0.000
9,F1,40,XGBoost,0.095,0.007,0.297,0.000


  ✓ portfolio formation uses pre-test information only | assets screened on the window before the test year; scores from frozen models and lagged features


## 11 · Descriptive ecosystem: communities, metadata audit, structural exposure, prospective validation

In [ ]:
# CELL 11.1 — Leiden communities, alignment with metadata, relation internality
import igraph as ig, leidenalg as la
from sklearn.metrics import normalized_mutual_info_score as NMI, adjusted_rand_score as ARI

def communities():
    g = ig.Graph(n=N, edges=UNION.tolist())
    part = la.find_partition(g, la.ModularityVertexPartition, seed=42)
    return np.array(part.membership), float(part.modularity)

COMM, MODQ = stage("communities", communities, deps=[len(UNION)])
align = []
for col, mask in [("asset_type", np.ones(N, bool)), ("exchange", ASSETS.exchange.map(valid).to_numpy()),
                  ("currency", ASSETS.currency.map(valid).to_numpy()), ("country", ASSETS.country.map(valid).to_numpy()),
                  ("sector", ASSETS.sector.map(valid).to_numpy()), ("fund_category", ASSETS.fund_category.map(valid).to_numpy())]:
    if mask.sum() > 10:
        lab = ASSETS[col].astype(str).to_numpy()[mask]
        align.append(dict(classification=col, assets=int(mask.sum()), NMI=NMI(lab, COMM[mask]), ARI=ARI(lab, COMM[mask])))
ALIGN = pd.DataFrame(align)
INTERN = pd.DataFrame([dict(relation=r, edges=len(E), within_pct=100 * float((COMM[E[:, 0]] == COMM[E[:, 1]]).mean()) if len(E) else np.nan)
                       for r, E in REL.items()])
COMP = pd.crosstab(COMM, ASSETS.asset_type, normalize="index").reset_index().rename(columns={"row_0": "community"})
save_csv(ALIGN, "community_alignment"); save_csv(INTERN, "relation_internality"); save_csv(COMP, "community_composition")
log(f"Leiden: {COMM.max() + 1} communities, modularity {MODQ:.4f}")
display(ALIGN.round(4)); display(INTERN.round(2))

2026-10-06 12:32:25 | [loaded from Drive] communities
2026-10-06 12:32:25 | Leiden: 11 communities, modularity 0.5453


,classification,assets,NMI,ARI
0,asset_type,5905,0.3304,0.1294
1,exchange,5905,0.7278,0.4025
2,currency,5864,0.7354,0.5516
3,country,3000,0.7148,0.6640
4,sector,2832,0.0695,0.0569
5,fund_category,1001,0.3157,0.0217


,relation,edges,within_pct
0,semantic_similarity,66294,60.17
1,same_sector,67169,23.02
2,same_fund_category,6273,96.60
3,same_country,64520,91.67
4,same_exchange,134913,93.82
5,same_currency,140208,70.05


In [ ]:
# CELL 11.2 — relation-balanced personalised PageRank (structural exposure, not contagion)
def exposure():
    W = {}
    for r, E in REL.items():
        for a_, b_ in E:
            W[(a_, b_)] = W.get((a_, b_), 0) + 1.0 / len(E)
    G = nx.Graph(); G.add_nodes_from(range(N)); G.add_weighted_edges_from([(a_, b_, w) for (a_, b_), w in W.items()])
    Gu = nx.Graph(); Gu.add_nodes_from(range(N)); Gu.add_edges_from(W.keys())
    pr = nx.pagerank(G, alpha=0.85, weight="weight")
    rows, mat = [], []
    for t in TYPES:
        cand = [i for i in range(N) if atype[i] == t]
        src = max(cand, key=lambda i: pr[i])
        ppr = nx.pagerank(G, alpha=0.85, personalization={src: 1.0}, weight="weight")
        ppu = nx.pagerank(Gu, alpha=0.85, personalization={src: 1.0})
        v = np.array([ppr[i] for i in range(N)]); v[src] = 0; v /= v.sum()
        u = np.array([ppu[i] for i in range(N)]); u[src] = 0; u /= u.sum()
        hhi = float((v ** 2).sum())
        rows.append(dict(source_class=t, source=ASSETS.symbol[src], cross_asset_mass=float(v[atype != t].sum()),
                         cross_community_mass=float(v[COMM != COMM[src]].sum()), effective_reach=1 / hhi,
                         spearman_vs_unweighted=float(sst.spearmanr(v, u).correlation)))
        mat.append({tt: float(v[atype == tt].sum()) for tt in TYPES} | {"source_class": t})
    return pd.DataFrame(rows), pd.DataFrame(mat)

EXPO, EXPO_MAT = stage("exposure", exposure, deps=[len(UNION)])
save_csv(EXPO, "structural_exposure"); save_csv(EXPO_MAT, "structural_exposure_matrix")
display(EXPO.round(4))

2026-10-06 12:32:25 | [loaded from Drive] exposure


,source_class,source,cross_asset_mass,cross_community_mass,effective_reach,spearman_vs_unweighted
0,Crypto,YEP-USD,0.6537,0.3952,1750.2738,0.9662
1,Currency,INRUSD=X,0.7327,0.4495,1263.5449,0.9475
2,ETF,FCPI,0.6484,0.2244,233.9189,0.9543
3,Equity,9478.T,0.1603,0.6677,2005.7130,0.9541
4,Fund,MUOIX,0.5626,0.2563,323.9787,0.9453
5,Index,^NQEMASIALMAUDN,0.6795,0.5775,1445.6776,0.8744


In [ ]:
# CELL 11.3 — do 2021 communities predict later realised correlation? (node-cluster bootstrap)
def community_validation():
    rows = []
    elig = np.concatenate([SPLIT[s] for s in SPLIT])
    rng = np.random.default_rng(99)
    for w in sorted({f["test"] for f in CFG["FOLDS"]}):
        C, n = pairwise_corr(window(R_DAILY, w)[elig])
        iu, ju = np.triu_indices(len(elig), 1)
        ok = n[iu, ju] >= CFG["MIN_OVERLAP_DAILY"]
        iu, ju = iu[ok], ju[ok]
        c = C[iu, ju]; same = COMM[elig[iu]] == COMM[elig[ju]]
        diff0 = c[same].mean() - c[~same].mean()
        boots = []
        for _ in range(CFG["BOOTSTRAP_REPS"]):
            cnt = np.bincount(rng.integers(0, len(elig), len(elig)), minlength=len(elig)).astype(float)
            wgt = cnt[iu] * cnt[ju]
            if wgt[same].sum() > 0 and wgt[~same].sum() > 0:
                boots.append(np.average(c[same], weights=wgt[same]) - np.average(c[~same], weights=wgt[~same]))
        rows.append(dict(window=w, pairs=len(c), within_mean=c[same].mean(), across_mean=c[~same].mean(), diff=diff0,
                         ci_l=np.percentile(boots, 2.5), ci_u=np.percentile(boots, 97.5)))
    return pd.DataFrame(rows)

COMMVAL = stage("community_validation", community_validation, deps=[len(UNION), DATA_SIG])
save_csv(COMMVAL, "community_validation"); display(COMMVAL.round(4))

2026-10-06 12:32:26 | [loaded from Drive] community_validation


,window,pairs,within_mean,across_mean,diff,ci_l,ci_u
0,t3,6492488,0.1847,0.0816,0.1031,0.0947,0.1123
1,t4,6418922,0.2100,0.0944,0.1157,0.1069,0.1250


## 12 · Bridge test: why did the earlier +0.057 AP gain disappear?
The earlier draft found that pairwise text + relation fusion beat lagged correlation by 0.057 AP. The strict
design above does not. This section moves from the strict design back towards the earlier one **one change at
a time**, re-running the same models on both test years, so each step's effect can be read off directly:

| Step | Change relative to the previous step |
|---|---|
| S0 | strict design (results above) |
| S1 | funds and indices removed from Task 2, as in the earlier draft (they stay in the graph); tuning as in S0 |
| S2 | earlier fixed hyperparameters instead of equal-budget tuning |
| S3 | earlier return calendar: each asset's own trading days (crypto weekends included) |

Every step keeps the leakage controls, the node-disjoint split and the locked test years.

In [ ]:
# CELL 12.1 — bridge steps (Study A only)
BRIDGE_CLASSES = ["Crypto", "Currency", "ETF", "Equity"]
BRIDGE_MODELS = ["Text cosine", "Lagged correlation", "Logistic", "Pair-only", "GAT"]
OLD_HP = {"Logistic": dict(C=1.0), "Pair-only": dict(lr=1e-3, hid=64, dropout=0.2), "GAT": dict(lr=1e-3, hid=64, dropout=0.2)}
BRIDGE_STEPS = [dict(code="B1", name="S1: no funds/indices in Task 2", tuned=True, calendar="business"),
                dict(code="B2", name="S2: + fixed earlier hyperparameters", tuned=False, calendar="business"),
                dict(code="B3", name="S3: + own-trading-day returns", tuned=False, calendar="own")]

def returns_own_calendar(P):
    """Earlier convention: log returns between each asset's consecutive observations, on the union of dates."""
    P = P.reindex(columns=ASSETS.yahoo).copy(); P.columns = ASSETS.idx.values
    R = pd.concat({c: np.log(P[c].dropna()).diff() for c in P.columns}, axis=1).reindex(P.index)
    R = R.replace([np.inf, -np.inf], np.nan)
    crypto = (ASSETS.asset_type == "Crypto").to_numpy()
    return R.mask(R.abs() > np.where(crypto, 1.0, 0.5))

def run_bridge():
    global SPLIT, GRAPHS, TYPE_PAIRS, TP_INDEX, FEATURE_GROUPS, R_DAILY, BEST_HP
    main_state = dict(SPLIT=SPLIT, GRAPHS=GRAPHS, TYPE_PAIRS=TYPE_PAIRS, TP_INDEX=TP_INDEX,
                      FEATURE_GROUPS=FEATURE_GROUPS, R_DAILY=R_DAILY, BEST_HP=BEST_HP)
    R_own = stage("bridge_returns_own", lambda: returns_own_calendar(PRICES), deps=[DATA_SIG])
    rows, boots, hps = [], [], []
    try:
        for step in BRIDGE_STEPS:
            R_DAILY = R_own if step["calendar"] == "own" else main_state["R_DAILY"]
            h0 = window(R_DAILY, "H0").notna().sum().to_numpy()
            def split_fn(h0=h0):
                elig = ASSETS.idx[(h0 >= CFG["ASSET_MIN_OBS_H0"]) & ASSETS.asset_type.isin(BRIDGE_CLASSES)].to_numpy()
                rng = np.random.default_rng(CFG["SPLIT_SEED"]); out = {"train": [], "val": [], "test": []}
                for t in BRIDGE_CLASSES:
                    m = elig[atype[elig] == t]; m = m[rng.permutation(len(m))]
                    a = int(round(CFG["SPLIT_FRACTIONS"][0] * len(m))); b = a + int(round(CFG["SPLIT_FRACTIONS"][1] * len(m)))
                    out["train"] += m[:a].tolist(); out["val"] += m[a:b].tolist(); out["test"] += m[b:].tolist()
                return {k: np.array(sorted(v), dtype=np.int64) for k, v in out.items()}
            SPLIT = stage(f"bridge_split_{step['calendar']}", split_fn, deps=[DATA_SIG, BRIDGE_CLASSES])
            audit(f"bridge {step['code']}: splits disjoint and limited to {'/'.join(BRIDGE_CLASSES)}",
                  not (set(SPLIT["train"]) & set(SPLIT["test"])) and set(atype[np.concatenate(list(SPLIT.values()))]) <= set(BRIDGE_CLASSES))
            GRAPHS = {s: make_graph(induced(SPLIT[s])) for s in ("train", "val", "test")}
            TYPE_PAIRS = list(itertools.combinations_with_replacement(sorted(BRIDGE_CLASSES), 2))
            TP_INDEX = {tp: k for k, tp in enumerate(TYPE_PAIRS)}
            FEATURE_GROUPS = {"Description cosine": [0], "Lagged correlation (+ missing flag)": [1, 2], "Same asset class": [3],
                              "Relation indicators": list(range(4, 4 + len(RELATIONS))),
                              "Asset-class-pair indicators": list(range(4 + len(RELATIONS), 4 + len(RELATIONS) + len(TYPE_PAIRS)))}
            for f in CFG["FOLDS"]:
                FOLD_DATA[(f["name"], step["code"])] = stage(f"bridge_pairs_{step['calendar']}_{f['name']}",
                                                              lambda f=f: build_fold(f, "D"), deps=[DATA_SIG, SPLIT["train"].tolist()])
            DATA_CACHE.clear()
            if step["tuned"]:
                st_ = ("F1", step["code"], CFG["TAU_MAIN"], "lag"); d = get_data(st_); BEST_HP = dict(main_state["BEST_HP"])
                for m in ("Logistic", "Pair-only", "GAT"):
                    res = [(hp, average_precision_score(d["y"]["val"], run_model(st_, m, hp, 42, d)["val"])) for hp in GRIDS[m]]
                    BEST_HP[m] = max(res, key=lambda x: x[1])[0]
                    hps.append(dict(step=step["code"], model=m, hp=json.dumps(BEST_HP[m]), val_AP=max(r[1] for r in res)))
            else:
                BEST_HP = dict(main_state["BEST_HP"], **OLD_HP)
            for f in CFG["FOLDS"]:
                st_ = (f["name"], step["code"], CFG["TAU_MAIN"], "lag")
                run_suite(st_, BRIDGE_MODELS, CFG["MODEL_SEEDS"])
                d = get_data(st_)
                for m in BRIDGE_MODELS:
                    aps = [r["row"]["AP"] for r in SCORES[(st_, m)].values()]
                    caps = [r["row"]["cross_AP"] for r in SCORES[(st_, m)].values()]
                    rows.append(dict(step=step["code"], step_name=step["name"], fold=f["name"], model=m, runs=len(aps),
                                     AP_mean=np.mean(aps), AP_sd=np.std(aps, ddof=1) if len(aps) > 1 else np.nan,
                                     cross_AP_mean=np.nanmean(caps), test_prevalence_pct=100 * d["y"]["test"].mean(),
                                     test_pairs=len(d["y"]["test"]), test_nodes=len(SPLIT["test"])))
                b = bootstrap_contrasts(st_, [("Logistic", "Lagged correlation"), ("Pair-only", "Lagged correlation"),
                                              ("GAT", "Pair-only")], CFG["BOOTSTRAP_REPS"])
                boots.append(b.assign(step=step["code"], step_name=step["name"]))
                log(f"bridge {step['code']} {f['name']} done")
    finally:   # always restore the strict-design state
        SPLIT, GRAPHS, TYPE_PAIRS, TP_INDEX = main_state["SPLIT"], main_state["GRAPHS"], main_state["TYPE_PAIRS"], main_state["TP_INDEX"]
        FEATURE_GROUPS, R_DAILY, BEST_HP = main_state["FEATURE_GROUPS"], main_state["R_DAILY"], main_state["BEST_HP"]
        DATA_CACHE.clear()
    return pd.DataFrame(rows), pd.concat(boots, ignore_index=True), pd.DataFrame(hps)

if RUN_BRIDGE:
    BR_SUM, BR_BOOT, BR_HP = stage("bridge_test", run_bridge, deps=[DATA_SIG, sorted(T2_RUNS.key.tolist())])
    s0 = main[main.model.isin(BRIDGE_MODELS)][["fold", "model", "runs", "AP_mean", "AP_sd", "cross_AP_mean"]].assign(
        step="S0", step_name="S0: strict design")
    s0b = BOOT[BOOT.apply(lambda r: (r.model, r.versus) in [("Logistic", "Lagged correlation"), ("Pair-only", "Lagged correlation"),
                                                            ("GAT", "Pair-only")], axis=1)].assign(step="S0", step_name="S0: strict design")
    BR_SUM = pd.concat([s0, BR_SUM], ignore_index=True); BR_BOOT = pd.concat([s0b, BR_BOOT], ignore_index=True)
    save_csv(BR_SUM, "bridge_summary"); save_csv(BR_BOOT, "bridge_contrasts"); save_csv(BR_HP, "bridge_hp_selected")
    view = BR_BOOT[["step", "fold", "model", "versus", "diff", "ci_l", "ci_u", "p_holm"]].round(4)
    display(view)
    display(BR_SUM.pivot_table(index=["step", "model"], columns="fold", values="AP_mean").round(4))
    audit("strict-design state restored after bridge test",
          len(TYPE_PAIRS) == len(TYPES) * (len(TYPES) + 1) // 2 and R_DAILY.index.dayofweek.max() <= 4,
          "split, graphs, features and calendar back to the strict design")

## 13 · Pre-registered tests: where should text help?
The strict design shows that text and graph do not beat lagged correlation **on average**. Theory says text
should matter most where price history is weak. The hypotheses, contrasts and decision rules below were
written **before** these analyses were run; the plan is saved with a timestamp to
`audits/preregistration.json` on first execution and never overwritten. Every result is reported, and Holm
adjustment is applied across all primary contrasts.

| ID | Hypothesis | Primary contrast (test AP, node-cluster bootstrap) | Supported if |
|---|---|---|---|
| H1 | **No price history (cold start):** text + relations rank future co-movement better than knowing only the asset classes | logistic on text + relations + class vs logistic on class only, lagged features removed | difference > 0 and Holm p < 0.05 |
| H1b | **Short history:** among pairs with < 120 days of overlapping history, fusion beats lagged correlation | logistic fusion − lagged correlation, within that subgroup | as H1; reported as under-powered if < 30 positives |
| H2 | **Longer horizon:** as the lag grows from 1 to 3 years, text's gain over a stale lagged correlation grows | (fusion − lagged) at 3-year lag minus the same at 1-year lag, test year 2024–25 | > 0 and Holm p < 0.05 |
| H3 | **Correlation change:** text + relations predict which pairs' correlation will rise by > 0.20 | logistic fusion vs logistic on lagged correlation only | > 0 and Holm p < 0.05 |
| H4 | **Duplicates (Study A):** adding back duplicate listings of the same instrument (crypto quotes in other currencies, cross-listings, identical descriptions) inflates text's apparent gain | (fusion − lagged) with duplicates minus without | > 0 and Holm p < 0.05 |

In [ ]:
# CELL 13.1 — record the plan before running anything
RUN_PREREG = True
PREREG = dict(
    written="before running section 13",
    H1="cold start: logistic(text+relations+class, no lag) − logistic(class only, no lag) > 0; folds F1, F2",
    H1b="short history (<120 overlapping days in lag window): logistic fusion − lagged correlation > 0; F1, F2",
    H2="lag gap 3y vs 1y, test t4: [fusion − lagged](gap 3) − [fusion − lagged](gap 1) > 0",
    H3="target 1[ρ_t − ρ_lag > 0.20]: logistic(full) − logistic(lag only) > 0; F1, F2",
    H4="Study A: [fusion − lagged](with duplicates) − [fusion − lagged](without) > 0; F1, F2",
    alpha=0.05, adjustment="Holm across all primary contrasts", bootstrap="node-cluster, test assets",
    reps=CFG["BOOTSTRAP_REPS"], seeds=CFG["SENS_SEEDS"])
PRE_PATH = DIR["audits"] / "preregistration.json"
if not PRE_PATH.exists():
    PREREG["timestamp"] = f"{datetime.now():%Y-%m-%d %H:%M:%S}"
    _atomic(PRE_PATH, lambda p: Path(p).write_text(json.dumps(PREREG, indent=2)))
log(f"pre-registration on file: {json.loads(PRE_PATH.read_text()).get('timestamp')}")

def boot_did(y, li, lj, n_nodes, arms, reps, seed=2718):
    """arms: name → (score vectors A, score vectors B, row mask or None). Returns the point estimate of
    mean-AP(A) − mean-AP(B) per arm and its node-cluster bootstrap draws (identical draws for every arm)."""
    prep, point = {}, {}
    for k, (A, B, m) in arms.items():
        m = np.ones(len(y), bool) if m is None else m
        prep[k] = ([FastAP(y[m], a[m]) for a in A], [FastAP(y[m], b[m]) for b in B], m)
        point[k] = np.mean([average_precision_score(y[m], a[m]) for a in A]) - np.mean([average_precision_score(y[m], b[m]) for b in B])
    rng = np.random.default_rng(seed)
    draws = {k: np.empty(reps) for k in arms}
    for r in range(reps):
        cnt = np.bincount(rng.integers(0, n_nodes, n_nodes), minlength=n_nodes).astype(float)
        w = cnt[li] * cnt[lj]
        for k, (FA, FB, m) in prep.items():
            draws[k][r] = np.nanmean([f(w[m]) for f in FA]) - np.nanmean([f(w[m]) for f in FB])
    return point, draws

def summarise_draws(point, draws):
    d = draws[np.isfinite(draws)]
    p = 2 * min((d <= 0).mean(), (d >= 0).mean()) if len(d) else np.nan
    return dict(diff=point, ci_l=np.percentile(d, 2.5), ci_u=np.percentile(d, 97.5), p_boot=min(1.0, max(p, 1 / max(len(d), 1))))

def custom_inputs(blocks, tau, keep=None, label=None, mask=None):
    """Task-2 inputs from arbitrary pair blocks (train/val/test), optional column subset, label and row mask."""
    pf_tr, st = pair_features(blocks["train"], "lag")
    data = dict(x={}, graph={}, pairs={}, pf={}, y={}, cross={}, extra={}, raw={})
    for s in ("train", "val", "test"):
        B = blocks[s]
        m = np.ones(len(B["i"]), bool) if mask is None else mask(B)
        pf = (pf_tr if s == "train" else pair_features(B, "lag", st)[0])[m]
        data["pf"][s] = pf if keep is None else pf[:, keep]
        data["y"][s] = ((B["target"] > tau) if label is None else label(B))[m].astype(np.float32)
        data["x"][s], data["graph"][s] = X_NODE[SPLIT[s]], GRAPHS[s]
        data["pairs"][s] = np.column_stack([B["li"], B["lj"]])[m]
        data["cross"][s] = (atype[B["i"]] != atype[B["j"]])[m]
        data["raw"][s] = dict(cos=rowdot(X_TEXT, B["i"], B["j"])[m], lag=B["lag"][m], lag_res=B["lag_res"][m])
    return data

PRE_ROWS, PRE_PRIMARY = [], []

2026-10-06 12:32:26 | pre-registration on file: 2026-10-06 12:11:52


In [ ]:
# CELL 13.2 — H1 (no history) and H1b (short history)
N_PF = 4 + len(RELATIONS) + len(TYPE_PAIRS)
NO_LAG = [c for c in range(N_PF) if c not in (1, 2)]
CLASS_ONLY = [3] + list(range(4 + len(RELATIONS), N_PF))

def h1():
    rows, prim = [], []
    for f in CFG["FOLDS"]:
        blocks = FOLD_DATA[(f["name"], "D")]
        res = {}
        for name, keep, model in (("NoLag fusion (logistic)", NO_LAG, "Logistic"), ("Class only (logistic)", CLASS_ONLY, "Logistic"),
                                  ("NoLag fusion (pair-only NN)", NO_LAG, "Pair-only")):
            d = custom_inputs(blocks, CFG["TAU_MAIN"], keep=keep)
            seeds = CFG["SENS_SEEDS"] if model == "Pair-only" else [42]
            tag = "nolag" if keep is NO_LAG else "classonly"
            res[name] = [run_model((f["name"], "D", CFG["TAU_MAIN"], tag), model, BEST_HP[model], s_, d)["test"] for s_ in seeds]
            ap = np.mean([average_precision_score(d["y"]["test"], x) for x in res[name]])
            rows.append(dict(hyp="H1", fold=f["name"], model=name, AP=ap, base_rate=d["y"]["test"].mean(), lift=ap / d["y"]["test"].mean()))
        y = d["y"]["test"]; li, lj = d["pairs"]["test"][:, 0], d["pairs"]["test"][:, 1]
        pt, dr = boot_did(y, li, lj, len(SPLIT["test"]),
                          {"logit": (res["NoLag fusion (logistic)"], res["Class only (logistic)"], None),
                           "nn": (res["NoLag fusion (pair-only NN)"], res["Class only (logistic)"], None)}, CFG["BOOTSTRAP_REPS"])
        prim.append(dict(hyp="H1", fold=f["name"], contrast="NoLag fusion (logistic) − Class only", primary=True, **summarise_draws(pt["logit"], dr["logit"])))
        prim.append(dict(hyp="H1", fold=f["name"], contrast="NoLag fusion (pair-only NN) − Class only", primary=False, **summarise_draws(pt["nn"], dr["nn"])))
    return rows, prim

def h1b():
    rows, prim = [], []
    for f in CFG["FOLDS"]:
        st_ = (f["name"], "D", CFG["TAU_MAIN"], "lag")
        d = get_data(st_); B = FOLD_DATA[(f["name"], "D")]["test"]
        if "lag_n" in B:
            ln = B["lag_n"]
        else:   # pair tables cached by an earlier notebook version: recompute overlap counts
            _, n_ = pairwise_corr(window(R_DAILY, CFG["PREV"][f["test"]])[SPLIT["test"]]); ln = n_[B["li"], B["lj"]]
        m = ln < 120
        y = d["y"]["test"]
        sc = {k: [r["test"] for r in SCORES[(st_, k)].values()] for k in ("Logistic", "Lagged correlation", "Pair-only")}
        npos = int(y[m].sum())
        rows.append(dict(hyp="H1b", fold=f["name"], subgroup_pairs=int(m.sum()), subgroup_positives=npos,
                         AP_logistic=average_precision_score(y[m], sc["Logistic"][0][m]) if npos else np.nan,
                         AP_lagged=average_precision_score(y[m], sc["Lagged correlation"][0][m]) if npos else np.nan,
                         underpowered=npos < 30))
        if npos >= 5:
            li, lj = d["pairs"]["test"][:, 0], d["pairs"]["test"][:, 1]
            pt, dr = boot_did(y, li, lj, len(SPLIT["test"]), {"a": (sc["Logistic"], sc["Lagged correlation"], m)}, CFG["BOOTSTRAP_REPS"])
            prim.append(dict(hyp="H1b", fold=f["name"], contrast="Logistic − Lagged (history < 120 days)", primary=True,
                             underpowered=npos < 30, **summarise_draws(pt["a"], dr["a"])))
        else:
            prim.append(dict(hyp="H1b", fold=f["name"], contrast="Logistic − Lagged (history < 120 days)", primary=True,
                             underpowered=True, diff=np.nan, ci_l=np.nan, ci_u=np.nan, p_boot=np.nan))
    return rows, prim

if RUN_PREREG:
    r1, p1 = stage("prereg_H1", h1, deps=[DATA_SIG, PREREG["H1"]])
    r1b, p1b = stage("prereg_H1b", h1b, deps=[DATA_SIG, PREREG["H1b"], sorted(T2_RUNS.key.tolist())])
    PRE_ROWS += r1 + r1b; PRE_PRIMARY += p1 + p1b
    display(pd.DataFrame(r1).round(4)); display(pd.DataFrame(r1b).round(4))

2026-10-06 12:32:26 | [loaded from Drive] prereg_H1
2026-10-06 12:32:27 | [loaded from Drive] prereg_H1b


,hyp,fold,model,AP,base_rate,lift
0,H1,F1,NoLag fusion (logistic),0.3677,0.0436,8.4415
1,H1,F1,Class only (logistic),0.2507,0.0436,5.7550
2,H1,F1,NoLag fusion (pair-only NN),0.3603,0.0436,8.2704
3,H1,F2,NoLag fusion (logistic),0.3125,0.0529,5.9111
4,H1,F2,Class only (logistic),0.2129,0.0529,4.0279
5,H1,F2,NoLag fusion (pair-only NN),0.3066,0.0529,5.8009


,hyp,fold,subgroup_pairs,subgroup_positives,AP_logistic,AP_lagged,underpowered
0,H1b,F1,722,0,NaN,NaN,True
1,H1b,F2,1515,3,0.3595,0.0525,True


In [ ]:
# CELL 13.3 — H2 (longer horizon) and H3 (correlation change)
def h2():
    lag_for = {1: ("t2", "t3"), 2: ("t1", "t2"), 3: ("H0", "t1")}    # (lag for t3 train/val, lag for t4 test)
    res, rows, keys = {}, [], None
    for g, (lag_tr, lag_te) in lag_for.items():
        blocks = {s: pair_block(SPLIT[s], "t3", R_DAILY, CFG["MIN_OVERLAP_DAILY"], lag_w=lag_tr) for s in ("train", "val")}
        blocks["test"] = pair_block(SPLIT["test"], "t4", R_DAILY, CFG["MIN_OVERLAP_DAILY"], lag_w=lag_te)
        k = blocks["test"]["i"] * N + blocks["test"]["j"]
        if keys is None:
            keys = k
        audit(f"H2 gap {g}: identical test pairs across gaps", np.array_equal(keys, k))
        d = custom_inputs(blocks, CFG["TAU_MAIN"])
        st_ = ("H2", f"gap{g}", CFG["TAU_MAIN"], "lag")
        res[g] = {m: [run_model(st_, m, BEST_HP.get(m), 42, d)["test"]] for m in ("Logistic", "Lagged correlation")}
        rows.append(dict(hyp="H2", gap_years=g, lag_window_test=lag_te,
                         AP_logistic=average_precision_score(d["y"]["test"], res[g]["Logistic"][0]),
                         AP_lagged=average_precision_score(d["y"]["test"], res[g]["Lagged correlation"][0])))
    y = d["y"]["test"]; li, lj = d["pairs"]["test"][:, 0], d["pairs"]["test"][:, 1]
    arms = {g: (res[g]["Logistic"], res[g]["Lagged correlation"], None) for g in (1, 3)}
    pt, dr = boot_did(y, li, lj, len(SPLIT["test"]), arms, CFG["BOOTSTRAP_REPS"])
    prim = [dict(hyp="H2", fold="t4", contrast="[Logistic − Lagged] gap 3 − gap 1", primary=True,
                 **summarise_draws(pt[3] - pt[1], dr[3] - dr[1]))]
    for g in (1, 3):
        prim.append(dict(hyp="H2", fold="t4", contrast=f"Logistic − Lagged at gap {g}", primary=False, **summarise_draws(pt[g], dr[g])))
    return rows, prim

def h3():
    rows, prim = [], []
    LAG_ONLY = [1, 2]
    label = lambda B: (B["target"] - B["lag"]) > 0.20
    mask = lambda B: np.isfinite(B["lag"]) & np.isfinite(B["target"])
    for f in CFG["FOLDS"]:
        blocks = FOLD_DATA[(f["name"], "D")]
        res = {}
        for name, keep in (("Full fusion", None), ("Lag only", LAG_ONLY)):
            d = custom_inputs(blocks, CFG["TAU_MAIN"], keep=keep, label=label, mask=mask)
            res[name] = [run_model((f["name"], "D", 0.20, "change_" + ("full" if keep is None else "lagonly")), "Logistic",
                                   BEST_HP["Logistic"], 42, d)["test"]]
        y = d["y"]["test"]; li, lj = d["pairs"]["test"][:, 0], d["pairs"]["test"][:, 1]
        rows.append(dict(hyp="H3", fold=f["name"], pairs=len(y), positives=int(y.sum()), base_rate=y.mean(),
                         AP_full=average_precision_score(y, res["Full fusion"][0]), AP_lagonly=average_precision_score(y, res["Lag only"][0])))
        pt, dr = boot_did(y, li, lj, len(SPLIT["test"]), {"a": (res["Full fusion"], res["Lag only"], None)}, CFG["BOOTSTRAP_REPS"])
        prim.append(dict(hyp="H3", fold=f["name"], contrast="Full fusion − Lag only (Δρ > 0.20)", primary=True, **summarise_draws(pt["a"], dr["a"])))
    return rows, prim

if RUN_PREREG:
    r2, p2 = stage("prereg_H2", h2, deps=[DATA_SIG, PREREG["H2"]])
    r3, p3 = stage("prereg_H3", h3, deps=[DATA_SIG, PREREG["H3"]])
    PRE_ROWS += r2 + r3; PRE_PRIMARY += p2 + p3
    display(pd.DataFrame(r2).round(4)); display(pd.DataFrame(r3).round(4))

2026-10-06 12:32:27 | [loaded from Drive] prereg_H2
2026-10-06 12:32:27 | [loaded from Drive] prereg_H3


,hyp,gap_years,lag_window_test,AP_logistic,AP_lagged
0,H2,1,t3,0.7030,0.7406
1,H2,2,t2,0.7283,0.7772
2,H2,3,t1,0.7236,0.7743


,hyp,fold,pairs,positives,base_rate,AP_full,AP_lagonly
0,H3,F1,261630,6803,0.0260,0.0623,0.1645
1,H3,F2,261370,22768,0.0871,0.1199,0.1523


In [ ]:
# CELL 13.4 — H4: do duplicate listings inflate text's apparent gain? (Study A only; self-contained)
DUP_CLASSES = ["Crypto", "ETF", "Equity"]
DUP_PER_ASSET, DUP_TOTAL = 2, (600 if not SMOKE else 30)

def find_duplicates():
    raw = RAW.copy()
    raw["name"] = np.where(raw.long_name.map(valid), raw.long_name, raw.short_name)
    raw = raw[raw.name.map(valid)].copy()
    raw["summary_clean"] = [clean_summary(x, c) for x, c in zip(raw.summary, raw.asset_type)]
    raw["key"] = np.where(raw.summary_clean.str.len() >= CFG["MIN_SUMMARY_CHARS"], raw.summary_clean.str.lower().str[:400], None)
    cr = raw.asset_type == "Crypto"
    raw.loc[cr, "coin_key"] = np.where(raw.loc[cr, "coin"].map(valid), raw.loc[cr, "coin"], raw.loc[cr, "symbol"].str.split("-").str[0])
    lookup = raw.drop_duplicates(["asset_type", "symbol"]).set_index(["asset_type", "symbol"])
    taken = set(ASSETS.symbol)
    rows = []
    for a in ASSETS[ASSETS.asset_type.isin(DUP_CLASSES)].itertuples():
        if (a.asset_type, a.symbol) not in lookup.index:
            continue
        me = lookup.loc[(a.asset_type, a.symbol)]
        if a.asset_type == "Crypto":
            c = raw[cr & (raw.coin_key == me.coin_key) & (raw.symbol != a.symbol)]
        elif me.key is not None:
            c = raw[(raw.asset_type == a.asset_type) & (raw.key == me.key) & (raw.symbol != a.symbol)]
        else:
            continue
        c = c[~c.symbol.isin(taken)].drop_duplicates("symbol")
        c = c.assign(r=c.symbol.map(stable_rank)).sort_values("r").head(DUP_PER_ASSET)
        for d in c.itertuples():
            rows.append(dict(symbol=d.symbol, asset_type=a.asset_type, orig_idx=a.idx, name=d.name,
                             text=(d.name + ". " + d.summary_clean) if len(d.summary_clean) >= CFG["MIN_SUMMARY_CHARS"] else d.name,
                             sector=d.sector, fund_category=d.fund_category, country=d.country, exchange=d.exchange,
                             currency=d.currency))
    dup = pd.DataFrame(rows).drop_duplicates("symbol")
    dup = dup.assign(r=dup.symbol.map(stable_rank)).sort_values("r").head(DUP_TOTAL).drop(columns="r").reset_index(drop=True)
    return dup

def encode_texts(texts, like=None):
    if SMOKE:   # identical-text duplicates get the original's vector plus tiny noise
        V = np.stack([X_TEXT[o] + np.random.default_rng(k).normal(0, 0.01, 768) for k, o in enumerate(like)])
        return (V / np.linalg.norm(V, axis=1, keepdims=True)).astype(np.float32)
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(CFG["TEXT_MODEL"], device=DEVICE)
    return m.encode(list(texts), batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

def h4():
    dup = find_duplicates()
    log(f"H4: {len(dup)} duplicate listings found ({dup.asset_type.value_counts().to_dict()})")
    if SMOKE:
        rng = np.random.default_rng(5)
        P_dup = pd.DataFrame({r.symbol: PRICES[ASSETS.yahoo[r.orig_idx]].to_numpy() * np.exp(np.cumsum(rng.normal(0, 0.003, len(PRICES))))
                              for r in dup.itertuples()}, index=PRICES.index)
        P_all = pd.concat([PRICES, P_dup], axis=1)
    else:
        download_prices(dup.symbol.tolist())
        P_all = load_price_panel()
    base = ASSETS[ASSETS.asset_type.isin(BRIDGE_CLASSES)]
    U = pd.concat([base.assign(orig_idx=base.idx, is_dup=False), dup.assign(yahoo=dup.symbol, is_dup=True)], ignore_index=True)
    U["u"] = np.arange(len(U))
    X = np.vstack([X_TEXT[base.idx.to_numpy()], encode_texts(dup.text, like=dup.orig_idx.to_numpy())])
    R = make_returns(P_all, "D", yahoo=U.yahoo, cols=U.u.to_numpy(), crypto=(U.asset_type == "Crypto").to_numpy())[0]
    h0 = window(R, "H0").notna().sum().to_numpy()
    elig = U.u[h0 >= CFG["ASSET_MIN_OBS_H0"]].to_numpy()
    rng = np.random.default_rng(CFG["SPLIT_SEED"]); spl = {"train": [], "val": [], "test": []}
    ut = U.asset_type.to_numpy()
    for t in BRIDGE_CLASSES:
        m_ = elig[ut[elig] == t]; m_ = m_[rng.permutation(len(m_))]
        a_ = int(round(CFG["SPLIT_FRACTIONS"][0] * len(m_))); b_ = a_ + int(round(CFG["SPLIT_FRACTIONS"][1] * len(m_)))
        spl["train"] += m_[:a_].tolist(); spl["val"] += m_[a_:b_].tolist(); spl["test"] += m_[b_:].tolist()
    spl = {k: np.array(sorted(v)) for k, v in spl.items()}
    meta = {c: U[c].astype(str).to_numpy() for c in ("sector", "fund_category", "country", "exchange", "currency")}
    okm = {c: U[c].map(valid).to_numpy() for c in meta}
    tps = list(itertools.combinations_with_replacement(sorted(BRIDGE_CLASSES), 2)); tpi = {t: k for k, t in enumerate(tps)}
    isdup = U.is_dup.to_numpy(); orig = U.orig_idx.to_numpy()

    def block(nodes, w):
        Ct, nt = pairwise_corr(window(R, w)[nodes]); Cl, nl = pairwise_corr(window(R, CFG["PREV"][w])[nodes])
        iu, ju = np.triu_indices(len(nodes), 1); k = nt[iu, ju] >= CFG["MIN_OVERLAP_DAILY"]
        iu, ju = iu[k], ju[k]; i, j = nodes[iu], nodes[ju]
        lag = np.where(nl[iu, ju] >= CFG["LAG_MIN_OVERLAP"], Cl[iu, ju], np.nan)
        F_ = np.zeros((len(i), 4 + 5 + len(tps)), np.float32)
        F_[:, 0] = rowdot(X, i, j); F_[:, 1] = lag; F_[:, 2] = ~np.isfinite(lag); F_[:, 3] = ut[i] == ut[j]
        for q, c in enumerate(meta):
            F_[:, 4 + q] = okm[c][i] & okm[c][j] & (meta[c][i] == meta[c][j])
        F_[np.arange(len(i)), 9 + np.array([tpi[tuple(sorted((x, y)))] for x, y in zip(ut[i], ut[j])], dtype=int)] = 1
        same_instr = (orig[i] == orig[j])           # a duplicate paired with its original or a sibling
        return dict(i=i, j=j, y=(Ct[iu, ju] > CFG["TAU_MAIN"]).astype(int), F=F_, lag=lag, same=same_instr,
                    base=~isdup[i] & ~isdup[j])

    rows, prim = [], []
    for f in CFG["FOLDS"]:
        B = {s: block(spl[s], f[s]) for s in ("train", "val", "test")}
        out = {}
        for arm in ("without duplicates", "with duplicates"):
            sel = {s: (B[s]["base"] if arm == "without duplicates" else np.ones(len(B[s]["y"]), bool)) for s in B}
            Ftr = B["train"]["F"][sel["train"]].copy(); med = np.nanmedian(Ftr[:, 1]); Ftr[:, 1] = np.nan_to_num(Ftr[:, 1], nan=med)
            mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
            prep = lambda s: (np.nan_to_num(B[s]["F"][sel[s]], nan=med) - mu) / sd
            clf = LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced").fit((Ftr - mu) / sd, B["train"]["y"][sel["train"]])
            te = sel["test"]
            out[arm] = dict(logit=clf.predict_proba(prep("test"))[:, 1], lag=np.nan_to_num(B["test"]["lag"][te], nan=med), mask=te)
            yt = B["test"]["y"][te]
            rows.append(dict(hyp="H4", fold=f["name"], arm=arm, test_pairs=int(te.sum()), positives=int(yt.sum()),
                             same_instrument_pairs=int(B["test"]["same"][te].sum()),
                             same_instrument_positive_rate=float(yt[B["test"]["same"][te]].mean()) if B["test"]["same"][te].any() else np.nan,
                             AP_logistic=average_precision_score(yt, out[arm]["logit"]), AP_lagged=average_precision_score(yt, out[arm]["lag"])))
        tn = spl["test"]; pos = {int(v): k for k, v in enumerate(tn)}
        li = np.array([pos[v] for v in B["test"]["i"]]); lj = np.array([pos[v] for v in B["test"]["j"]])
        y = B["test"]["y"]
        fa = {arm: (FastAP(y[o["mask"]], o["logit"]), FastAP(y[o["mask"]], o["lag"]), o["mask"]) for arm, o in out.items()}
        pts = {arm: average_precision_score(y[o["mask"]], o["logit"]) - average_precision_score(y[o["mask"]], o["lag"]) for arm, o in out.items()}
        rng = np.random.default_rng(31); draws = {a: [] for a in out}
        for _ in range(CFG["BOOTSTRAP_REPS"]):
            cnt = np.bincount(rng.integers(0, len(tn), len(tn)), minlength=len(tn)).astype(float); w = cnt[li] * cnt[lj]
            for arm, (A, Bb, m) in fa.items():
                draws[arm].append(A(w[m]) - Bb(w[m]))
        dA, dB = np.array(draws["with duplicates"]), np.array(draws["without duplicates"])
        prim.append(dict(hyp="H4", fold=f["name"], contrast="[Logistic − Lagged] with − without duplicates", primary=True,
                         **summarise_draws(pts["with duplicates"] - pts["without duplicates"], dA - dB)))
        for arm, d_ in (("with duplicates", dA), ("without duplicates", dB)):
            prim.append(dict(hyp="H4", fold=f["name"], contrast=f"Logistic − Lagged ({arm})", primary=False, **summarise_draws(pts[arm], d_)))
    return rows, prim, dup

if RUN_PREREG and STUDY == "A":
    r4, p4, DUPS = stage("prereg_H4", h4, deps=[DATA_SIG, PREREG["H4"], DUP_TOTAL])
    PRE_ROWS += r4; PRE_PRIMARY += p4
    save_csv(DUPS, "prereg_H4_duplicate_listings")
    display(pd.DataFrame(r4).round(4))

In [ ]:
# CELL 13.5 — Holm across primary contrasts; one table for the manuscript
if RUN_PREREG:
    PR = pd.DataFrame(PRE_PRIMARY)
    prim = PR[PR.primary & PR.p_boot.notna()].copy()
    order = np.argsort(prim.p_boot.to_numpy()); m_ = len(prim); adj = np.empty(m_); run = 0
    for k, idx in enumerate(order):
        run = max(run, min(1.0, (m_ - k) * prim.p_boot.iloc[idx])); adj[idx] = run
    prim["p_holm"] = adj
    PR = PR.merge(prim[["hyp", "fold", "contrast", "p_holm"]], on=["hyp", "fold", "contrast"], how="left")
    PR["supported"] = PR.primary & (PR["diff"] > 0) & (PR.p_holm < 0.05)
    save_csv(PR, "prereg_contrasts"); save_csv(pd.DataFrame(PRE_ROWS), "prereg_details")
    display(PR[["hyp", "fold", "contrast", "primary", "diff", "ci_l", "ci_u", "p_holm", "supported"]].round(4))
    audit("pre-registered plan written before results", PRE_PATH.exists(), json.loads(PRE_PATH.read_text()).get("timestamp"))

,hyp,fold,contrast,primary,diff,ci_l,ci_u,p_holm,supported
0,H1,F1,NoLag fusion (logistic) − Class only,True,0.1170,0.0879,0.1495,0.005,True
1,H1,F1,NoLag fusion (pair-only NN) − Class only,False,0.1096,0.0854,0.1364,NaN,False
2,H1,F2,NoLag fusion (logistic) − Class only,True,0.0995,0.0783,0.1231,0.005,True
3,H1,F2,NoLag fusion (pair-only NN) − Class only,False,0.0937,0.0745,0.1138,NaN,False
4,H1b,F1,Logistic − Lagged (history < 120 days),True,NaN,NaN,NaN,NaN,False
5,H1b,F2,Logistic − Lagged (history < 120 days),True,NaN,NaN,NaN,NaN,False
6,H2,t4,[Logistic − Lagged] gap 3 − gap 1,True,-0.0131,-0.0289,0.0020,0.084,False
7,H2,t4,Logistic − Lagged at gap 1,False,-0.0377,-0.0522,-0.0232,NaN,False
8,H2,t4,Logistic − Lagged at gap 3,False,-0.0507,-0.0731,-0.0302,NaN,False
9,H3,F1,Full fusion − Lag only (Δρ > 0.20),True,-0.1023,-0.1620,-0.0672,0.005,False


  ✓ pre-registered plan written before results | 2026-10-06 12:11:52


## 14 · External datasets: Fama–French factors, SEC N-PORT holdings, SEC 10-K business text
Three public datasets test the conclusions from new angles. Each downloads automatically and is cached on
Drive (`shared/external/`); if a download fails, that extension is skipped with a message in
`results/external_status.csv` and the rest of the notebook still finishes.

| ID | Data | Question | Primary contrast |
|---|---|---|---|
| E1 | Fama–French daily factors (US + Developed, 5 factors + momentum) | Does lagged co-movement survive standard factor residualisation? | robustness, reported with bootstrap CI |
| E2a | SEC Form N-PORT fund/ETF holdings, filed before the snapshot | Do holdings links add to lagged correlation? | logistic(+holdings) − logistic; F1, F2 |
| E2b | same | …in the cold start? | same without lag; F1, F2 |
| E2c | same | With real cross-asset links, does message passing finally help? | GAT on holdings graph − pair-only with holdings features; F1, F2 |
| E3 | SEC EDGAR 10-K Item 1 (latest filed before 31 Jul 2021) | Does the cold-start text result hold with dated, point-in-time text? | logistic(10-K + FinanceDatabase text) − logistic(FinanceDatabase text), cold start, US equities; F1, F2 |

E2a–E2c and E3 primaries form one Holm family. The plan is saved to `audits/preregistration_ext.json`
before anything runs. E2 version 2 (`audits/preregistration_ext_E2v2.json`) matches held securities by ticker,
CUSIP and company name (v1 used tickers only and linked just 12 test pairs in Study A) and adds a
common-ownership relation: two securities held together by several funds.

In [ ]:
# CELL 14.1 — configuration and plan
RUN_EXT = True
EXT = dict(
    SEC_USER_AGENT="KG research tanzinayeasmin@bubt.edu.bd",     # SEC requires a contact e-mail in the User-Agent
    FF_FILES=["F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", "F-F_Momentum_Factor_daily_CSV.zip",
              "Developed_5_Factors_Daily_CSV.zip", "Developed_Mom_Factor_Daily_CSV.zip"],
    FF_BASE="https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/",
    NPORT_QUARTERS=["2021q1", "2021q2", "2021q3"],          # filing quarters; filings after the snapshot are dropped
    NPORT_URLS=["https://www.sec.gov/files/dera/data/form-n-port-data-sets/{q}_nport.zip",
                "https://www.sec.gov/files/structureddata/data/form-n-port-data-sets/{q}_nport.zip"],
    NPORT_TOP=25, COHOLD_MIN=5, HOLD_TOP=100, COOWN_MIN=3, COOWN_MAX_HOLDINGS=400, K_COOWN=15,
    TENK_MIN_CHARS=500, TENK_MAX_CHARS=5000,
    TENK_MIN_COMPANIES=10 if SMOKE else 50, TENK_MIN_POSITIVES=2 if SMOKE else 10,
)
EXT_DIR = DIR["shared"] / "external"
(EXT_DIR / "nport").mkdir(parents=True, exist_ok=True); (EXT_DIR / "edgar").mkdir(parents=True, exist_ok=True)
PREREG_EXT = dict(written="before running section 14",
                  E1="robustness: AP of Fama–French-residual lagged correlation and of logistic fusion using it, vs raw",
                  E2a="AP(logistic + holdings) − AP(logistic) > 0; F1, F2", E2b="same, cold start (no lag); F1, F2",
                  E2c="AP(GAT, holdings graph + features) − AP(pair-only, holdings features) > 0; F1, F2",
                  E3="cold start, US equities with 10-K text: AP(logistic, FinanceDatabase + 10-K cosine) − AP(logistic, FinanceDatabase cosine) > 0; F1, F2",
                  holm="E2a, E2b, E2c, E3 primaries together", alpha=0.05)
_pe = DIR["audits"] / "preregistration_ext.json"
if not _pe.exists():
    PREREG_EXT["timestamp"] = f"{datetime.now():%Y-%m-%d %H:%M:%S}"
    _atomic(_pe, lambda p: Path(p).write_text(json.dumps(PREREG_EXT, indent=2)))
# E2 version 2: better security matching and a common-ownership relation. Written before E2 v2 runs in this
# study; for a study whose E2 v1 already ran, the timestamp shows that v2 was specified after v1 results.
PREREG_E2V2 = dict(version="E2v2", written="before E2 v2 runs in this study",
                   change="held securities matched by ticker, then CUSIP→ticker majority vote across all N-PORT holdings, then unique "
                          "company name; fund series → tickers from SEC company_tickers_mf.json; holds uses each fund's top 100 positions",
                   new_relation="co_owned: two universe securities both held by ≥3 funds that hold ≤400 universe securities "
                                "(Antón–Polk common ownership); pair features log(1+count) and indicator; graph keeps 15 strongest partners per asset",
                   E2a=PREREG_EXT["E2a"], E2b=PREREG_EXT["E2b"], E2c=PREREG_EXT["E2c"],
                   features="holds, co_holding, co_owned indicator, log(1 + co_owned count)", holm=PREREG_EXT["holm"])
_pe2 = DIR["audits"] / "preregistration_ext_E2v2.json"
if not _pe2.exists():
    PREREG_E2V2["timestamp"] = f"{datetime.now():%Y-%m-%d %H:%M:%S}"
    _atomic(_pe2, lambda p: Path(p).write_text(json.dumps(PREREG_E2V2, indent=2)))
EXT_STATUS, EXT_PRIMARY, EXT_ROWS = [], [], []
import requests, io, html as _html

def http_get(url, sec=False, timeout=60):
    h = {"User-Agent": EXT["SEC_USER_AGENT"], "Accept-Encoding": "gzip, deflate"} if sec else {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=h, timeout=timeout)
    r.raise_for_status()
    return r

def ext_run(name, fn):
    """Run one extension; a failure is recorded and does not stop the notebook."""
    try:
        out = fn()
        EXT_STATUS.append(dict(extension=name, status="ok", detail=""))
        return out
    except Exception as e:
        log(f"[{name}] skipped: {type(e).__name__}: {e}")
        EXT_STATUS.append(dict(extension=name, status="skipped", detail=f"{type(e).__name__}: {e}"[:500]))
        return None

def append_cols(d, blocks, fn, mask=None):
    """Append extra pair columns fn(block) (standardised on training pairs) to custom_inputs data d."""
    cols = {}
    for s in ("train", "val", "test"):
        m = np.ones(len(blocks[s]["i"]), bool) if mask is None else mask(blocks[s])
        cols[s] = np.asarray(fn(blocks[s]), dtype=np.float64).reshape(len(blocks[s]["i"]), -1)[m]
    mu, sd = cols["train"].mean(0), cols["train"].std(0) + 1e-6
    for s in cols:
        d["pf"][s] = np.hstack([d["pf"][s], ((cols[s] - mu) / sd).astype(np.float32)])
    return d

In [ ]:
# CELL 14.2 — E1: Fama–French factor residuals
def ff_factors():
    path = EXT_DIR / "ff_daily.parquet"
    if path.exists():
        return pd.read_parquet(path)
    if SMOKE:
        idx = R_DAILY.index; rng = np.random.default_rng(11)
        F_ = pd.DataFrame(rng.normal(0, 0.01, (len(idx), 6)), index=idx, columns=["US_Mkt-RF", "US_SMB", "US_HML", "US_Mom", "DEV_Mkt-RF", "DEV_Mom"])
    else:
        frames = []
        for fname in EXT["FF_FILES"]:
            z = zipfile.ZipFile(io.BytesIO(http_get(EXT["FF_BASE"] + fname).content))
            raw = z.read(z.namelist()[0]).decode("latin-1").splitlines()
            start = next(k for k, l in enumerate(raw) if re.match(r"^\s*,", l))        # header row starts with a comma
            rows = []
            for l in raw[start + 1:]:
                p = [x.strip() for x in l.split(",")]
                if not re.match(r"^\d{8}$", p[0]):
                    break
                rows.append(p)
            cols = [c.strip() for c in raw[start].split(",")[1:]]
            df = pd.DataFrame(rows, columns=["date"] + cols).set_index("date").astype(float) / 100.0
            df.index = pd.to_datetime(df.index, format="%Y%m%d")
            tag = "DEV" if fname.startswith("Developed") else "US"
            frames.append(df.drop(columns=[c for c in df.columns if c == "RF"]).add_prefix(tag + "_"))
        F_ = pd.concat(frames, axis=1)
    _atomic(path, lambda p: F_.to_parquet(p))
    return F_

def residualise_ff(Rw, F_):
    Fw = F_.reindex(Rw.index).fillna(0.0).to_numpy()
    Fm = np.column_stack([Fw, np.ones(len(Fw))])
    X = Rw.to_numpy(float); out = np.full_like(X, np.nan)
    for j in range(X.shape[1]):
        ok = np.isfinite(X[:, j])
        if ok.sum() > Fm.shape[1] + 10:
            beta, *_ = np.linalg.lstsq(Fm[ok], X[ok, j], rcond=None)
            out[ok, j] = X[ok, j] - Fm[ok] @ beta
    return pd.DataFrame(out, index=Rw.index, columns=Rw.columns)

def e1():
    F_ = ff_factors()
    rows, prim = [], []
    for f in CFG["FOLDS"]:
        blocks = {}
        for s in ("train", "val", "test"):
            B = dict(FOLD_DATA[(f["name"], "D")][s])
            C, n_ = pairwise_corr(residualise_ff(window(R_DAILY, CFG["PREV"][f[s]])[SPLIT[s]], F_))
            B["lag"] = np.where(n_[B["li"], B["lj"]] >= CFG["LAG_MIN_OVERLAP"], C[B["li"], B["lj"]], np.nan)
            blocks[s] = B
        d_ff = custom_inputs(blocks, CFG["TAU_MAIN"])
        d_raw = get_data((f["name"], "D", CFG["TAU_MAIN"], "lag"))
        res = {"FF-residual lagged correlation": [run_model((f["name"], "D", CFG["TAU_MAIN"], "ff"), "Lagged correlation", None, 42, d_ff)["test"]],
               "Logistic, FF-residual lag": [run_model((f["name"], "D", CFG["TAU_MAIN"], "ff"), "Logistic", BEST_HP["Logistic"], 42, d_ff)["test"]],
               "Lagged correlation (raw)": [r["test"] for r in SCORES[((f["name"], "D", CFG["TAU_MAIN"], "lag"), "Lagged correlation")].values()],
               "Logistic, raw lag": [r["test"] for r in SCORES[((f["name"], "D", CFG["TAU_MAIN"], "lag"), "Logistic")].values()][:1]}
        y = d_raw["y"]["test"]; li, lj = d_raw["pairs"]["test"][:, 0], d_raw["pairs"]["test"][:, 1]
        for k, v in res.items():
            rows.append(dict(ext="E1", fold=f["name"], model=k, AP=average_precision_score(y, v[0])))
        pt, dr = boot_did(y, li, lj, len(SPLIT["test"]), {"det": (res["FF-residual lagged correlation"], res["Lagged correlation (raw)"], None),
                                                          "log": (res["Logistic, FF-residual lag"], res["Logistic, raw lag"], None)}, CFG["BOOTSTRAP_REPS"])
        prim.append(dict(hyp="E1", fold=f["name"], contrast="FF-residual lag − raw lag", primary=False, **summarise_draws(pt["det"], dr["det"])))
        prim.append(dict(hyp="E1", fold=f["name"], contrast="Logistic: FF-residual lag − raw lag", primary=False, **summarise_draws(pt["log"], dr["log"])))
    return rows, prim

if RUN_EXT:
    out = ext_run("E1 Fama–French", lambda: stage("ext_E1", e1, deps=[DATA_SIG, PREREG_EXT["E1"]]))
    if out:
        EXT_ROWS += out[0]; EXT_PRIMARY += out[1]; display(pd.DataFrame(out[0]).round(4))

2026-10-06 12:35:56 | [computed + saved] ext_E1 (208s)


,ext,fold,model,AP
0,E1,F1,FF-residual lagged correlation,0.3101
1,E1,F1,"Logistic, FF-residual lag",0.4905
2,E1,F1,Lagged correlation (raw),0.8633
3,E1,F1,"Logistic, raw lag",0.8662
4,E1,F2,FF-residual lagged correlation,0.2592
5,E1,F2,"Logistic, FF-residual lag",0.3652
6,E1,F2,Lagged correlation (raw),0.7406
7,E1,F2,"Logistic, raw lag",0.7183


In [ ]:
# CELL 14.3 — E2 (v2): SEC N-PORT holdings → "holds", "co-holding" and "co-owned" relations
def _col(df, *keys):
    for c in df.columns:
        if all(k in c.upper() for k in keys):
            return c
    raise KeyError(f"no column with {keys} in {list(df.columns)[:12]}")

def norm_co(s):
    """Company name key for matching N-PORT issuer names to FinanceDatabase names."""
    s = re.sub(r"[^a-z0-9 ]+", " ", str(s or "").lower())
    drop = {"inc", "incorporated", "corp", "corporation", "co", "company", "ltd", "limited", "plc", "the", "holdings",
            "holding", "group", "sa", "nv", "ag", "se", "llc", "lp", "class", "cl", "a", "b", "com", "common", "stock", "new"}
    return " ".join(w for w in s.split() if w not in drop)

def nport_quarter(q, zpath):
    """Every holding in one N-PORT quarter that can identify a listed security (study-independent, cached)."""
    out = EXT_DIR / "nport" / f"{q}_extract_v2.parquet"
    if out.exists():
        return pd.read_parquet(out)
    z = zipfile.ZipFile(zpath); names = {Path(n).stem.upper(): n for n in z.namelist()}
    sub = pd.read_csv(z.open(names["SUBMISSION"]), sep="\t", dtype=str)
    info = pd.read_csv(z.open(names["FUND_REPORTED_INFO"]), sep="\t", dtype=str)
    sub = sub.assign(fdate=pd.to_datetime(sub[_col(sub, "FILING", "DATE")], errors="coerce"),
                     rdate=pd.to_datetime(sub[_col(sub, "REPORT")], errors="coerce"))
    acc = (info[[_col(info, "ACCESSION"), _col(info, "SERIES", "ID")]].set_axis(["acc", "series"], axis=1)
           .merge(sub[[_col(sub, "ACCESSION"), "fdate", "rdate"]].set_axis(["acc", "fdate", "rdate"], axis=1), on="acc"))
    acc = acc[acc.series.notna()]
    tick = {}
    for ch in pd.read_csv(z.open(names["IDENTIFIERS"]), sep="\t", dtype=str, chunksize=1_000_000):
        tc = next((c for c in ch.columns if "TICKER" in c.upper()), None)
        if tc is None:
            break
        ch = ch[ch[tc].notna()]
        tick.update(zip(ch[_col(ch, "HOLDING", "ID")], ch[tc].str.upper().str.strip()))
    acc_set, keep = set(acc.acc), []
    for ch in pd.read_csv(z.open(names["FUND_REPORTED_HOLDING"]), sep="\t", dtype=str, chunksize=1_000_000):
        ch = ch[ch[_col(ch, "ACCESSION")].isin(acc_set)]
        cat_c = next((c for c in ch.columns if "ASSET" in c.upper() and "CAT" in c.upper()), None)
        cus = next((c for c in ch.columns if "CUSIP" in c.upper()), None)
        nm = next((c for c in ch.columns if "ISSUER" in c.upper() and "NAME" in c.upper()), None)
        t = ch[_col(ch, "HOLDING", "ID")].map(tick)
        cat = ch[cat_c].fillna("") if cat_c else pd.Series("", index=ch.index)
        m = t.notna() | cat.isin(["EC", "EP", "RF"])
        keep.append(pd.DataFrame({"acc": ch.loc[m, _col(ch, "ACCESSION")], "ticker": t[m],
                                  "cusip": ch.loc[m, cus].str.upper().str.strip() if cus else None,
                                  "name": ch.loc[m, nm] if nm else None, "cat": cat[m],
                                  "pct": pd.to_numeric(ch.loc[m, _col(ch, "PERCENT")], errors="coerce")}))
    H = pd.concat(keep, ignore_index=True).merge(acc, on="acc")
    _atomic(out, lambda p: H.to_parquet(p, index=False))
    log(f"N-PORT {q}: {len(H):,} security holdings from {H.series.nunique():,} fund series")
    return H

def smoke_nport():
    rng = np.random.default_rng(9)
    eq = ASSETS[ASSETS.asset_type == "Equity"]; funds = ASSETS[ASSETS.asset_type.isin(["ETF", "Fund"])].symbol.tolist()
    grp = {g: d.symbol.tolist() for g, d in eq.groupby(eq.sector.fillna("x"))}
    rows, series_sym = [], {}
    nm_of = dict(zip(eq.symbol, eq.name))
    for k in range(120):
        sid = f"S{k:05d}"
        if k < len(funds):
            series_sym[sid] = [funds[k]]
        pool = grp[list(grp)[k % len(grp)]] + list(rng.choice(eq.symbol, 5))
        for r, h in enumerate(rng.choice(pool, min(30, len(pool)), replace=False)):
            nmh = nm_of.get(h, h)
            rows.append(dict(series=sid, ticker=h if rng.random() < 0.5 else None, cusip=f"C{stable_rank(h) % 10**8:08d}",
                             name=nmh, cat="EC", pct=float(30 - r)))
    H = pd.DataFrame(rows)
    H.loc[H.ticker.isna() & (rng.random(len(H)) < 0.5), "name"] = None      # some only resolvable via CUSIP votes
    return H, series_sym

def nport_latest(H):
    """Keep each series' latest filing on or before the snapshot; fill missing tickers from CUSIP votes."""
    snap = pd.Timestamp(CFG["GRAPH_SNAPSHOT_DATE"])
    H = H[(H.fdate <= snap) & (H.rdate <= snap)]
    # CUSIP → ticker by majority vote over every pre-snapshot holding that reports both
    cv = (H[H.ticker.notna() & H.cusip.notna() & (H.cusip.str.len() >= 8)].groupby(["cusip", "ticker"]).size()
          .reset_index(name="n").sort_values("n", ascending=False).drop_duplicates("cusip"))
    last = H[["series", "acc", "fdate", "rdate"]].drop_duplicates().sort_values(["fdate", "rdate", "acc"]).groupby("series").acc.last()
    H = H[H.acc.isin(set(last))].reset_index(drop=True)
    H["ticker"] = H.ticker.fillna(H.cusip.map(dict(zip(cv.cusip, cv.ticker))))
    return H[["series", "ticker", "cusip", "name", "cat", "pct"]]

def nport_raw():
    """Latest pre-snapshot N-PORT filing per fund series, held securities resolved to tickers where possible."""
    path, mpath = EXT_DIR / "nport_latest_v2.parquet", EXT_DIR / "nport_series_symbols.json"
    if path.exists() and mpath.exists():
        return pd.read_parquet(path), json.loads(mpath.read_text())
    if SMOKE:
        H, series_sym = smoke_nport()
    else:
        mf = http_get("https://www.sec.gov/files/company_tickers_mf.json", sec=True).json()
        mfd = pd.DataFrame(mf["data"], columns=mf["fields"])
        series_sym = mfd.groupby("seriesId").symbol.apply(lambda x: sorted(set(map(str, x)))).to_dict()
        frames = []
        for q in EXT["NPORT_QUARTERS"]:
            zpath = EXT_DIR / "nport" / f"{q}_nport.zip"
            if not zpath.exists():
                for u in EXT["NPORT_URLS"]:
                    try:
                        r = http_get(u.format(q=q), sec=True, timeout=900)
                        _atomic(zpath, lambda p, c=r.content: Path(p).write_bytes(c)); break
                    except Exception as e:
                        log(f"N-PORT {q}: {u.format(q=q)} failed ({e})")
            if not zpath.exists():
                raise FileNotFoundError(f"N-PORT {q} not downloadable. Download '{q}_nport.zip' from the SEC 'Form N-PORT Data Sets' page and put it in {EXT_DIR / 'nport'}")
            frames.append(nport_quarter(q, zpath))
        H = nport_latest(pd.concat(frames, ignore_index=True))
    _atomic(path, lambda p: H.to_parquet(p, index=False))
    mpath.write_text(json.dumps(series_sym))
    return H, series_sym

def resolve_holdings(H, series_sym):
    """Map held securities and fund series onto this study's universe (ticker, then CUSIP vote, then unique name)."""
    us = ASSETS[~ASSETS.symbol.str.contains(r"[.\-=^]", regex=True)]
    sym = dict(zip(us.symbol, us.idx))
    eq = us[us.asset_type == "Equity"].assign(k=lambda d: d.name.map(norm_co))
    eq = eq[(eq.k.str.len() >= 3) & ~eq.k.duplicated(keep=False)]
    name_map = dict(zip(eq.k, eq.idx))
    held = H.ticker.map(sym)
    how = np.where(held.notna(), "ticker", "")
    need = held.isna() & H.cat.isin(["EC", "EP"]) & H.name.notna()
    if need.any():
        un = {n: name_map.get(norm_co(n)) for n in H.loc[need, "name"].unique()}
        byname = H.loc[need, "name"].map(un)
        held.loc[need] = byname
        how[need.to_numpy() & byname.notna().reindex(H.index, fill_value=False).to_numpy()] = "name"
    R = H.assign(held=held, how=how, key=H.cusip.fillna(H.ticker).fillna(H.name.map(norm_co)))
    R["funds"] = R.series.map(lambda s_: [sym[x] for x in series_sym.get(s_, []) if x in sym])
    stats = dict(fund_series=int(H.series.nunique()), holdings=len(H),
                 universe_funds_matched=int(len({f_ for fl in R.drop_duplicates("series").funds for f_ in fl})),
                 holdings_resolved_to_universe=int(R.held.notna().sum()),
                 resolved_by_ticker_or_cusip=int((R.how == "ticker").sum()), resolved_by_name=int((R.how == "name").sum()),
                 universe_assets_held=int(R.held.nunique()))
    return R, stats

def holdings_relations(R):
    """holds (universe fund → universe security in its top HOLD_TOP), co_holding (two universe funds sharing
    ≥ COHOLD_MIN of their top NPORT_TOP positions), co_owned (two universe securities held by ≥ COOWN_MIN
    non-index-sized funds; Antón–Polk common ownership; graph keeps each asset's K_COOWN strongest partners)."""
    R = R.sort_values("pct", ascending=False)
    top = R.groupby("series").head(EXT["HOLD_TOP"])
    top = top[top.held.notna()]
    holds = canon([(int(f_), int(h)) for fl, h in zip(top.funds, top.held) for f_ in fl if f_ != h])
    t25 = R.groupby("series").head(EXT["NPORT_TOP"])
    tops = {}
    for fl, k in zip(t25.funds, t25.key):
        for f_ in fl:
            tops.setdefault(int(f_), set()).add(k)
    fl_ = sorted(tops)
    co = canon([(a, b) for i, a in enumerate(fl_) for b in fl_[i + 1:] if len(tops[a] & tops[b]) >= EXT["COHOLD_MIN"]])
    import scipy.sparse as sp
    E_ = R[R.held.notna()].drop_duplicates(["series", "held"])
    size = E_.groupby("series").held.transform("size")
    E_ = E_[(size >= 2) & (size <= EXT["COOWN_MAX_HOLDINGS"])]
    sid = E_.series.astype("category").cat.codes.to_numpy()
    A = sp.csr_matrix((np.ones(len(E_)), (sid, E_.held.astype(int).to_numpy())), shape=(int(sid.max()) + 1 if len(sid) else 0, N))
    C = sp.triu(A.T @ A, k=1).tocoo()
    ok = C.data >= EXT["COOWN_MIN"]
    counts = dict(zip((C.row[ok].astype(np.int64) * N + C.col[ok]).tolist(), C.data[ok].tolist()))
    cnt_df = pd.DataFrame({"i": C.row[ok], "j": C.col[ok], "c": C.data[ok]})
    both = pd.concat([cnt_df, cnt_df.rename(columns={"i": "j", "j": "i"})])
    knn = both.sort_values("c", ascending=False).groupby("i").head(EXT["K_COOWN"])
    coown = canon(knn[["i", "j"]].to_numpy()) if len(knn) else np.zeros((0, 2), np.int64)
    return {"holds": holds, "co_holding": co, "co_owned": coown}, counts

def e2():
    H, series_sym = nport_raw()
    R, rstats = resolve_holdings(H, series_sym)
    log(f"E2 N-PORT resolution: {rstats}")
    RH, coown_counts = holdings_relations(R)
    cser = pd.Series(coown_counts, dtype=float)
    keyset = {r: set((E[:, 0] * N + E[:, 1]).tolist()) for r, E in RH.items()}
    stats = [dict(relation=r, edges=len(E), cross_asset=int((atype[E[:, 0]] != atype[E[:, 1]]).sum()) if len(E) else 0) for r, E in RH.items()]
    stats.append(dict(relation="co_owned pairs (≥ COOWN_MIN funds, all)", edges=len(coown_counts), cross_asset=np.nan))
    log(f"E2 holdings relations: {stats}")
    def hcols(B):
        k = np.minimum(B["i"], B["j"]) * N + np.maximum(B["i"], B["j"])
        c = cser.reindex(k).fillna(0).to_numpy() if len(cser) else np.zeros(len(k))
        return np.column_stack([np.isin(k, list(keyset["holds"])), np.isin(k, list(keyset["co_holding"])), c > 0, np.log1p(c)])
    REL_H = dict(REL, **RH)
    rows, prim = [], []
    for f in CFG["FOLDS"]:
        blocks = FOLD_DATA[(f["name"], "D")]
        sc = {}
        for lagtag, keep in (("lag", None), ("nolag", NO_LAG)):
            base = custom_inputs(blocks, CFG["TAU_MAIN"], keep=keep)
            plus = append_cols(custom_inputs(blocks, CFG["TAU_MAIN"], keep=keep), blocks, hcols)
            sc[("base", lagtag)] = [run_model((f["name"], "D", CFG["TAU_MAIN"], f"e2base_{lagtag}"), "Logistic", BEST_HP["Logistic"], 42, base)["test"]]
            sc[("hold", lagtag)] = [run_model((f["name"], "D", CFG["TAU_MAIN"], f"e2v2hold_{lagtag}"), "Logistic", BEST_HP["Logistic"], 42, plus)["test"]]
            if lagtag == "lag":
                sc[("pair", lagtag)] = [run_model((f["name"], "D", CFG["TAU_MAIN"], "e2v2hold_lag"), "Pair-only", BEST_HP["Pair-only"], s_, plus)["test"] for s_ in CFG["SENS_SEEDS"]]
                for s in ("train", "val", "test"):
                    plus["graph"][s] = make_graph(induced(SPLIT[s], REL_H))
                sc[("gat", lagtag)] = [run_model((f["name"], "D", CFG["TAU_MAIN"], "e2v2holdgraph_lag"), "GAT", BEST_HP["GAT"], s_, plus)["test"] for s_ in CFG["SENS_SEEDS"]]
        y = base["y"]["test"]; li, lj = base["pairs"]["test"][:, 0], base["pairs"]["test"][:, 1]
        hc = hcols(blocks["test"]); hk = hc[:, :3].any(1); ck = hc[:, 2] > 0
        rows.append(dict(ext="E2", fold=f["name"], test_pairs=len(y), pairs_with_holdings_link=int(hk.sum()),
                         pairs_co_owned=int(ck.sum()), positives_with_link=int(y[hk].sum()),
                         positive_rate_with_link=float(y[hk].mean()) if hk.any() else np.nan,
                         positive_rate_co_owned=float(y[ck].mean()) if ck.any() else np.nan, positive_rate_all=float(y.mean()),
                         **{f"AP_{a}_{b}": average_precision_score(y, np.mean(v, 0)) for (a, b), v in sc.items()}))
        arms = {"a": (sc[("hold", "lag")], sc[("base", "lag")], None), "b": (sc[("hold", "nolag")], sc[("base", "nolag")], None),
                "c": (sc[("gat", "lag")], sc[("pair", "lag")], None)}
        if ck.any() and y[ck].sum() > 0 and (~y[ck].astype(bool)).sum() > 0:
            arms["d"] = (sc[("hold", "nolag")], sc[("base", "nolag")], ck)
        pt, dr = boot_did(y, li, lj, len(SPLIT["test"]), arms, CFG["BOOTSTRAP_REPS"])
        prim.append(dict(hyp="E2a", fold=f["name"], contrast="Logistic: +holdings − base", primary=True, **summarise_draws(pt["a"], dr["a"])))
        prim.append(dict(hyp="E2b", fold=f["name"], contrast="Cold start logistic: +holdings − base", primary=True, **summarise_draws(pt["b"], dr["b"])))
        prim.append(dict(hyp="E2c", fold=f["name"], contrast="GAT on holdings graph − pair-only (both with holdings features)", primary=True, **summarise_draws(pt["c"], dr["c"])))
        if "d" in arms:
            prim.append(dict(hyp="E2b", fold=f["name"], contrast="Cold start logistic: +holdings − base (co-owned pairs only)", primary=False, **summarise_draws(pt["d"], dr["d"])))
    return rows, prim, pd.DataFrame(stats), pd.DataFrame([rstats])

if RUN_EXT:
    out = ext_run("E2 N-PORT holdings", lambda: stage("ext_E2_v2", e2, deps=[DATA_SIG, PREREG_E2V2["version"], EXT["NPORT_QUARTERS"],
                                                                          {k: EXT[k] for k in ("NPORT_TOP", "COHOLD_MIN", "HOLD_TOP", "COOWN_MIN", "COOWN_MAX_HOLDINGS", "K_COOWN")}]))
    if out:
        EXT_ROWS += out[0]; EXT_PRIMARY += out[1]
        save_csv(out[2], "external_E2_holdings_relations"); save_csv(out[3], "external_E2_resolution")
        display(out[3]); display(out[2]); display(pd.DataFrame(out[0]).round(4))

2026-10-06 12:36:13 | E2 N-PORT resolution: {'fund_series': 6616, 'holdings': 1168943, 'universe_funds_matched': 293, 'holdings_resolved_to_universe': 100693, 'resolved_by_ticker_or_cusip': 92856, 'resolved_by_name': 7837, 'universe_assets_held': 816}
2026-10-06 12:36:14 | E2 holdings relations: [{'relation': 'holds', 'edges': 1613, 'cross_asset': 1604}, {'relation': 'co_holding', 'edges': 1580, 'cross_asset': 774}, {'relation': 'co_owned', 'edges': 8051, 'cross_asset': 304}, {'relation': 'co_owned pairs (≥ COOWN_MIN funds, all)', 'edges': 116571, 'cross_asset': nan}]
2026-10-06 12:40:36 | [run done] T2_2d66b2_F1_D_tau0.50_e2v2hold_lag_Pair-only_890368_s42 | test AP 0.8421 | epoch 8
2026-10-06 12:44:40 | [run done] T2_2d66b2_F1_D_tau0.50_e2v2hold_lag_Pair-only_890368_s7 | test AP 0.8542 | epoch 10
2026-10-06 12:48:33 | [run done] T2_2d66b2_F1_D_tau0.50_e2v2hold_lag_Pair-only_890368_s123 | test AP 0.8444 | epoch 10
2026-10-06 12:55:45 | [run done] T2_2d66b2_F1_D_tau0.50_e2v2holdgraph_la

,fund_series,holdings,universe_funds_matched,holdings_resolved_to_universe,resolved_by_ticker_or_cusip,resolved_by_name,universe_assets_held
0,6616,1168943,293,100693,92856,7837,816


,relation,edges,cross_asset
0,holds,1613,1604.0
1,co_holding,1580,774.0
2,co_owned,8051,304.0
3,"co_owned pairs (≥ COOWN_MIN funds, all)",116571,NaN


,ext,fold,test_pairs,pairs_with_holdings_link,pairs_co_owned,positives_with_link,positive_rate_with_link,positive_rate_co_owned,positive_rate_all,AP_base_lag,AP_hold_lag,AP_pair_lag,AP_gat_lag,AP_base_nolag,AP_hold_nolag
0,E2,F1,261630,2211,2087,146,0.0660,0.0235,0.0436,0.8662,0.8657,0.8479,0.8341,0.3677,0.3728
1,E2,F2,262085,2211,2087,231,0.1045,0.0570,0.0529,0.7183,0.7183,0.7225,0.7398,0.3125,0.3206


In [ ]:
# CELL 14.4 — E3: SEC 10-K Item 1 (Business), latest 10-K filed before the snapshot
def item1_text(raw_html):
    t = _html.unescape(re.sub(r"<[^>]+>", " ", raw_html))
    t = re.sub(r"\s+", " ", t)
    starts = [m.end() for m in re.finditer(r"item\s*1\s*\.?\s*[-–—:]?\s*business", t, re.I)]
    ends = [m.start() for m in re.finditer(r"item\s*(1a\s*\.?\s*[-–—:]?\s*risk\s*factors|2\s*\.?\s*[-–—:]?\s*properties)", t, re.I)]
    best = ""
    for s_ in starts:       # table-of-contents entries give short segments; the real section is the longest
        e_ = next((e for e in ends if e > s_), None)
        seg = t[s_:e_] if e_ else t[s_:s_ + 30000]
        if len(seg) > len(best):
            best = seg
    return best.strip(" .:-–—")

def tenk_texts():
    us = ASSETS[(ASSETS.asset_type == "Equity") & ~ASSETS.symbol.str.contains(r"[.\-=^]", regex=True)]
    if SMOKE:   # exercise the code path on every equity of the tiny smoke universe
        eq = ASSETS[ASSETS.asset_type == "Equity"]
        return {s: "Business. " + t + " " * 500 for s, t in zip(eq.symbol, eq.text)}
    cmap = {v["ticker"].upper(): int(v["cik_str"]) for v in http_get("https://www.sec.gov/files/company_tickers.json", sec=True).json().values()}
    out = {}
    for k, s in enumerate(us.symbol):
        cache = EXT_DIR / "edgar" / f"{s}.json"
        if cache.exists():
            out[s] = json.loads(cache.read_text()).get("text", ""); continue
        text, meta = "", {}
        try:
            cik = cmap.get(s)
            if cik:
                sub = http_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json", sec=True).json()["filings"]["recent"]
                cand = [(d, a, p) for f_, d, a, p in zip(sub["form"], sub["filingDate"], sub["accessionNumber"], sub["primaryDocument"])
                        if f_ in ("10-K", "10-K405") and d <= CFG["GRAPH_SNAPSHOT_DATE"]]
                if cand:
                    d, a, p = max(cand)
                    doc = http_get(f"https://www.sec.gov/Archives/edgar/data/{cik}/{a.replace('-', '')}/{p}", sec=True).text
                    text = item1_text(doc)[:EXT["TENK_MAX_CHARS"]]
                    meta = dict(cik=cik, filing_date=d, accession=a)
        except Exception as e:
            meta = dict(error=str(e)[:200])
        _atomic(cache, lambda p_, t=text, m=meta: Path(p_).write_text(json.dumps(dict(text=t, **m))))
        out[s] = text
        time.sleep(0.15)                       # SEC fair-access limit: ≤10 requests per second
        if k % 100 == 0:
            log(f"10-K: {k}/{len(us)} companies")
    return out

def e3():
    T = tenk_texts()
    ok = {s: t for s, t in T.items() if len(t) >= EXT["TENK_MIN_CHARS"]}
    if len(ok) < EXT["TENK_MIN_COMPANIES"]:
        raise RuntimeError(f"only {len(ok)} US equities with usable 10-K text")
    sym_idx = dict(zip(ASSETS.symbol, ASSETS.idx))
    idx = np.array([sym_idx[s] for s in ok])
    if SMOKE:
        V = X_TEXT[idx] + np.random.default_rng(4).normal(0, 0.3, (len(idx), 768))
        V = V / np.linalg.norm(V, axis=1, keepdims=True)
    else:
        from sentence_transformers import SentenceTransformer
        V = SentenceTransformer(CFG["TEXT_MODEL"], device=DEVICE).encode(list(ok.values()), batch_size=32, normalize_embeddings=True)
    X10 = np.zeros((N, 768), np.float32); X10[idx] = V; has = np.zeros(N, bool); has[idx] = True
    both = lambda B: has[B["i"]] & has[B["j"]]
    cos10 = lambda B: rowdot(X10, B["i"], B["j"])
    rows, prim = [], []
    for f in CFG["FOLDS"]:
        blocks = FOLD_DATA[(f["name"], "D")]
        a = custom_inputs(blocks, CFG["TAU_MAIN"], keep=NO_LAG, mask=both)
        b = append_cols(custom_inputs(blocks, CFG["TAU_MAIN"], keep=NO_LAG, mask=both), blocks, cos10, mask=both)
        if a["y"]["test"].sum() < EXT["TENK_MIN_POSITIVES"] or a["y"]["train"].sum() < EXT["TENK_MIN_POSITIVES"]:
            raise RuntimeError(f"too few positive pairs among 10-K-matched equities in {f['name']}")
        sa = [run_model((f["name"], "D", CFG["TAU_MAIN"], "e3_fdb"), "Logistic", BEST_HP["Logistic"], 42, a)["test"]]
        sb = [run_model((f["name"], "D", CFG["TAU_MAIN"], "e3_fdb10k"), "Logistic", BEST_HP["Logistic"], 42, b)["test"]]
        Bt = blocks["test"]; m = both(Bt)
        y = a["y"]["test"]; li, lj = Bt["li"][m], Bt["lj"][m]
        pos = np.unique(np.r_[li, lj]); remap = {v: k for k, v in enumerate(pos)}
        li2, lj2 = np.array([remap[v] for v in li]), np.array([remap[v] for v in lj])
        r_ = np.corrcoef(rowdot(X_TEXT, Bt["i"][m], Bt["j"][m]), cos10(Bt)[m])[0, 1]
        rows.append(dict(ext="E3", fold=f["name"], companies_with_10k=len(ok), test_pairs=int(m.sum()), positives=int(y.sum()),
                         AP_fdb=average_precision_score(y, sa[0]), AP_fdb_plus_10k=average_precision_score(y, sb[0]),
                         corr_fdb_vs_10k_similarity=r_))
        pt, dr = boot_did(y, li2, lj2, len(pos), {"a": (sb, sa, None)}, CFG["BOOTSTRAP_REPS"])
        prim.append(dict(hyp="E3", fold=f["name"], contrast="Cold start: FinanceDatabase + 10-K − FinanceDatabase", primary=True, **summarise_draws(pt["a"], dr["a"])))
    return rows, prim

if RUN_EXT:
    out = ext_run("E3 SEC 10-K", lambda: stage("ext_E3", e3, deps=[DATA_SIG, PREREG_EXT["E3"]]))
    if out:
        EXT_ROWS += out[0]; EXT_PRIMARY += out[1]; display(pd.DataFrame(out[0]).round(4))

2026-10-06 14:01:05 | 10-K: 0/793 companies
2026-10-06 14:02:28 | 10-K: 200/793 companies
2026-10-06 14:04:23 | 10-K: 500/793 companies
2026-10-06 14:05:06 | 10-K: 600/793 companies


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-10-06 14:16:59 | [computed + saved] ext_E3 (956s)


,ext,fold,companies_with_10k,test_pairs,positives,AP_fdb,AP_fdb_plus_10k,corr_fdb_vs_10k_similarity
0,E3,F1,294,1953,41,0.0609,0.0725,0.6647
1,E3,F2,294,1953,89,0.1240,0.1438,0.6647


In [ ]:
# CELL 14.5 — Holm across external-data primaries; status and tables
if RUN_EXT:
    save_csv(pd.DataFrame(EXT_STATUS), "external_status"); display(pd.DataFrame(EXT_STATUS))
    if EXT_PRIMARY:
        XP = pd.DataFrame(EXT_PRIMARY)
        pr = XP[XP.primary & XP.p_boot.notna()].copy()
        if len(pr):
            o = np.argsort(pr.p_boot.to_numpy()); adj = np.empty(len(pr)); run = 0
            for k, i in enumerate(o):
                run = max(run, min(1.0, (len(pr) - k) * pr.p_boot.iloc[i])); adj[i] = run
            pr["p_holm"] = adj
            XP = XP.merge(pr[["hyp", "fold", "contrast", "p_holm"]], on=["hyp", "fold", "contrast"], how="left")
        else:
            XP["p_holm"] = np.nan
        XP["supported"] = XP.primary & (XP["diff"] > 0) & (XP.p_holm < 0.05)
        save_csv(XP, "external_contrasts"); save_csv(pd.DataFrame(EXT_ROWS), "external_details")
        display(XP[["hyp", "fold", "contrast", "primary", "diff", "ci_l", "ci_u", "p_holm", "supported"]].round(4))

,extension,status,detail
0,E1 Fama–French,ok,
1,E2 N-PORT holdings,ok,
2,E3 SEC 10-K,ok,


,hyp,fold,contrast,primary,diff,ci_l,ci_u,p_holm,supported
0,E1,F1,FF-residual lag − raw lag,False,-0.5532,-0.6122,-0.4855,NaN,False
1,E1,F1,Logistic: FF-residual lag − raw lag,False,-0.3757,-0.4293,-0.3226,NaN,False
2,E1,F2,FF-residual lag − raw lag,False,-0.4815,-0.5398,-0.4173,NaN,False
3,E1,F2,Logistic: FF-residual lag − raw lag,False,-0.3531,-0.3956,-0.3099,NaN,False
4,E2a,F1,Logistic: +holdings − base,True,-0.0005,-0.0013,0.0004,0.804,False
5,E2b,F1,Cold start logistic: +holdings − base,True,0.0050,-0.0009,0.0125,0.480,False
6,E2c,F1,GAT on holdings graph − pair-only (both with h...,True,-0.0215,-0.0383,-0.0062,0.036,False
7,E2b,F1,Cold start logistic: +holdings − base (co-owne...,False,0.0450,-0.0489,0.1838,NaN,False
8,E2a,F2,Logistic: +holdings − base,True,-0.0000,-0.0009,0.0010,0.896,False
9,E2b,F2,Cold start logistic: +holdings − base,True,0.0081,0.0031,0.0145,0.008,True


## 15 · Figures and the results bundle

In [ ]:
# CELL 12.1 — figures (no titles; captions belong in the manuscript)
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42})
BLUE, ORANGE, VERM = "#0072B2", "#E69F00", "#D55E00"

def savefig(fig, name):
    for ext in ("pdf", "png"):
        _atomic(DIR["figures"] / f"{name}.{ext}", lambda p, e=ext: fig.savefig(p, format=e, dpi=300, bbox_inches="tight"))
    plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 2.8))
for k, m in enumerate(TASK1_MODELS):
    v = T1_RUNS[T1_RUNS.model == m].AP.to_numpy()
    ax.scatter(v, np.full(len(v), k) + np.random.default_rng(k).uniform(-.15, .15, len(v)), s=14, color=BLUE)
    ax.plot([v.mean()] * 2, [k - .3, k + .3], color="k")
lr_ap = T1_BASE.set_index("model").loc["Logistic regression", "AP"]
ax.axvline(lr_ap, ls=":", color="grey"); ax.set_yticks(range(len(TASK1_MODELS))); ax.set_yticklabels(TASK1_MODELS)
ax.set_xlabel("Task 1 test AP per seed (bar = mean; dotted = logistic regression)")
savefig(fig, "task1_seeds")

for f in CFG["FOLDS"]:
    sub = main[main.fold == f["name"]].sort_values("AP_mean")
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    y_ = np.arange(len(sub))
    ax.errorbar(sub.AP_mean, y_, xerr=sub.AP_sd.fillna(0), fmt="o", color=BLUE, label="all pairs")
    ax.plot(sub.cross_AP_mean, y_, "s", color=VERM, label="cross-asset pairs")
    ax.set_yticks(y_); ax.set_yticklabels(sub.model); ax.set_xlabel(f"Locked-test AP, fold {f['name']} (test {f['test']})")
    ax.legend(frameon=False, fontsize=7)
    savefig(fig, f"task2_ladder_{f['name']}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
lab = [f"{r.fold}: {r.model} − {r.versus}" for r in BOOT.itertuples()]
ax.errorbar(BOOT["diff"], range(len(BOOT)), xerr=[BOOT["diff"] - BOOT.ci_l, BOOT.ci_u - BOOT["diff"]], fmt="o", color=BLUE)
ax.axvline(0, color="grey", lw=.8); ax.set_yticks(range(len(BOOT))); ax.set_yticklabels(lab, fontsize=6.5)
ax.set_xlabel("Difference in test AP (node-cluster bootstrap 95% CI)")
savefig(fig, "task2_bootstrap_forest")
log("figures saved")

2026-10-06 14:17:03 | figures saved


In [ ]:
# CELL 12.2 — results bundle (send this zip back for the manuscript update)
summary = dict(study=STUDY, assets=N, union_edges=len(UNION), eligible=int(sum(len(v) for v in SPLIT.values())),
               test_prevalence={r.fold: r.prevalence_pct for r in T2_SAMPLE[(T2_SAMPLE.split == "test") & (T2_SAMPLE.freq == "D")].itertuples()},
               finished=f"{datetime.now():%Y-%m-%d %H:%M}", config_signature=CFG_SIG)
(DIR["results"] / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
zpath = ROOT / f"RESULTS_study_{STUDY}.zip"
def _zip(p):
    with zipfile.ZipFile(p, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in ("results", "figures", "audits"):
            for f in sorted(DIR[folder].glob("*")):
                z.write(f, f"study_{STUDY}/{folder}/{f.name}")
_atomic(zpath, _zip)
log(f"ALL DONE — results bundle: {zpath}")
print(json.dumps(summary, indent=2, default=str))
print(pd.DataFrame(_AUDITS.values())[["check", "passed"]].to_string(index=False))

2026-10-06 14:17:05 | ALL DONE — results bundle: /content/drive/MyDrive/KG_Strong_Pipeline/study_B/RESULTS_study_B.zip
{
  "study": "B",
  "assets": 5905,
  "union_edges": 432138,
  "eligible": 3685,
  "test_prevalence": {
    "F1": 4.356151817452127,
    "F2": 5.28607131274205
  },
  "finished": "2026-10-06 14:17",
  "config_signature": "3c6cc5290d"
}
                                                        check  passed
                                     asset identifiers unique    True
                                  one listing per crypto coin    True
                      descriptions de-duplicated within class    True
                              embeddings finite and unit-norm    True
               no relation-defining metadata in node features    True
                                  graph built without returns    True
                            returns use business-day calendar    True
                               node splits mutually exclusive    True
               